# Rapport d'adversaire Serie A 2025/26

Objectif : produire, pour chacune des 20 équipes de Serie A, un rapport d'adversaire lisible par un staff, à partir de chiffres uniquement. Aucune vidéo, aucun avis subjectif : chaque phrase du rapport vient d'un calcul, et chaque chiffre est comparé aux 19 autres équipes (moyenne, médiane, rang, percentile).

Le rapport suit trois niveaux, qu'il ne faut pas mélanger :
1. **Les données** : ce que les chiffres montrent.
2. **L'analyse** : ce que ces chiffres veulent dire, par rapport à la ligue.
3. **Les pistes à tester** : des idées liées à un chiffre précis, jamais un plan de match complet.

Sources : Understat (matchs, tirs, style de jeu) et Sofascore (statistiques d'équipe et de joueurs, compositions). Saison 2025/26 uniquement.

Chaque ligne de code est commentée, pour qu'une autre personne puisse reprendre le travail. Les réglages sont placés au début de chaque partie, à l'endroit où ils servent.

**Comment lire ce notebook.** Les parties 1 à 3 préparent les données de toute la ligue. Les parties suivantes construisent les analyses, puis la fonction qui produit le rapport d'une équipe. La dernière partie génère les 20 rapports et le site.

## 1. Réglages et imports

Un seul réglage est à connaître : `EQUIPE`, l'équipe d'exemple (nom exact tel qu'Understat l'écrit). Elle sert aux contrôles et aux affichages de démonstration, le rapport des 20 équipes est généré plus loin.

Prérequis : `pip install understatapi pandas numpy`. La collecte Sofascore (partie 4) demande aussi `playwright`.

In [1]:
import os  # pour créer les dossiers de sauvegarde s'ils n'existent pas encore
import json  # pour lire et écrire des fichiers JSON (format utilisé par le site)
import time  # pour faire de petites pauses entre les demandes au site
import pandas as pd  # pour manipuler les données sous forme de tableaux
import numpy as np  # pour les calculs sur les colonnes (comparaisons, moyennes)
from understatapi import UnderstatClient  # pour interroger Understat directement en Python

EQUIPE = "AC Milan"  # équipe d'exemple, nom exact tel qu'Understat l'écrit
SAISON = 2025  # année de départ de la saison, 2025 pour 2025/26

os.makedirs('data/raw', exist_ok=True)  # crée le dossier des données brutes, ne fait rien s'il existe déjà
os.makedirs('data/processed', exist_ok=True)  # crée le dossier des données nettoyées, ne fait rien s'il existe déjà

understat = UnderstatClient()  # crée le client qui va faire les requêtes vers Understat


def fr(nombre, decimales=2):  # petite recette : écrit un nombre avec une virgule, comme en français
    return f"{nombre:.{decimales}f}".replace('.', ',')  # arrondit au bon nombre de décimales puis remplace le point par une virgule

## 2. Les matchs des 20 équipes (Understat)

On récupère d'abord l'historique de match de chaque équipe : buts, points, xG (qualité des occasions), pressing. Tout le reste du notebook s'appuie sur ces tableaux.

**Contrôle attendu** : 20 équipes, et pour l'exemple de Milan 5e du classement avec 70 points.

In [2]:
league_table = understat.league(league="Serie_A").get_team_data(season=str(SAISON))  # récupère toutes les équipes de Serie A pour la saison choisie, str() car Understat attend du texte
print("Nombre d'équipes :", len(league_table))  # contrôle : on attend 20

noms_understat = [info['title'] for info in league_table.values()]  # liste des noms d'équipes tels qu'Understat les écrit
if EQUIPE not in noms_understat:  # si l'équipe d'exemple n'existe pas
    raise ValueError(f"Équipe introuvable : {EQUIPE}. Noms disponibles : {noms_understat}")  # arrête avec la liste des noms possibles

Nombre d'équipes : 20


In [3]:
tous = []  # liste vide qui accueillera le tableau de matchs de chaque équipe
for team_id, info in league_table.items():  # parcourt les 20 équipes
    h = pd.DataFrame(info['history'])  # transforme l'historique de matchs de l'équipe en tableau
    h['equipe'] = info['title']  # ajoute une colonne avec le nom de l'équipe, pour savoir de qui vient chaque ligne
    tous.append(h)  # range ce tableau dans la liste

tous_matchs = pd.concat(tous, ignore_index=True)  # colle les 20 tableaux les uns sous les autres pour n'en faire qu'un seul
tous_matchs['date'] = pd.to_datetime(tous_matchs['date'])  # convertit la date en vraie date

for col in ['xG', 'xGA', 'scored', 'missed', 'pts']:  # colonnes qui doivent être des nombres
    tous_matchs[col] = pd.to_numeric(tous_matchs[col])  # convertit en nombres, au cas où Understat les ait renvoyées en texte

classement = tous_matchs.groupby('equipe').agg(  # range les matchs par équipe et calcule des totaux pour chacune
    points=('pts', 'sum'),  # additionne les points gagnés
    buts_pour=('scored', 'sum'),  # additionne les buts marqués
    buts_contre=('missed', 'sum'),  # additionne les buts encaissés
)  # fin du calcul des totaux

classement['difference'] = classement['buts_pour'] - classement['buts_contre']  # différence de buts, qui sert à départager les égalités de points
classement = classement.sort_values(['points', 'difference'], ascending=False)  # trie par points puis par différence de buts, du meilleur au moins bon
classement['rang'] = range(1, len(classement) + 1)  # numérote les équipes de 1 à 20 dans l'ordre du tri

print("Rang de l'équipe :", classement.loc[EQUIPE, 'rang'], "avec", classement.loc[EQUIPE, 'points'], "points")  # contrôle : on attend 5 et 70 pour l'AC Milan
classement  # affiche le classement complet

Rang de l'équipe : 5 avec 70 points


,points,buts_pour,buts_contre,difference,rang
equipe,,,,,
Inter,87,89,35,54,1
Napoli,76,58,36,22,2
Roma,73,59,31,28,3
Como,71,65,29,36,4
AC Milan,70,53,35,18,5
Juventus,69,61,34,27,6
Atalanta,59,51,36,15,7
Bologna,56,49,46,3,8
Lazio,54,41,40,1,9


## 3. Style de jeu de chaque équipe : le PPDA

Le PPDA mesure l'intensité du pressing : c'est le nombre de passes que l'adversaire réussit avant que l'équipe tente de récupérer le ballon. **Plus il est bas, plus l'équipe presse fort et haut.** Pour étiqueter les équipes, on utilise deux seuils :
1. en dessous de 11 : pressing haut ;
2. de 11 à 13 : pressing médian ;
3. à partir de 13 : bloc bas.

Ces seuils sont un choix de lecture, pas une règle officielle. On l'indique au lecteur du rapport. On calcule aussi les actions que l'adversaire réussit près de la surface de l'équipe (`deep_subi_par_match`), un indicateur de la pression subie.

**Contrôle attendu** : 8 équipes en bloc bas, 7 en pressing haut et 5 en pressing médian. Milan : bloc bas, PPDA 14,62.

In [4]:
SEUIL_PRESSING_HAUT = 11  # en dessous de ce PPDA, l'équipe est classée en pressing haut
SEUIL_BLOC_BAS = 13  # à partir de ce PPDA, l'équipe est classée en bloc bas

lignes = []  # liste vide qui accueillera une ligne de résumé par équipe

for team_id, info in league_table.items():  # parcourt les 20 équipes de la Serie A
    h = pd.DataFrame(info['history'])  # transforme l'historique de matchs de l'équipe en tableau
    lignes.append({  # ajoute une ligne de résumé pour cette équipe
        'equipe': info['title'],  # nom de l'équipe
        'ppda': sum(x['att'] for x in h['ppda']) / sum(x['def'] for x in h['ppda']),  # PPDA de la saison, on additionne d'abord tout puis on divise, plus juste que de faire la moyenne des 38 rapports
        'ppda_subi': sum(x['att'] for x in h['ppda_allowed']) / sum(x['def'] for x in h['ppda_allowed']),  # même calcul pour le pressing que les adversaires font subir à l'équipe
        'deep_par_match': pd.to_numeric(h['deep']).mean(),  # moyenne des actions réussies près de la surface adverse
        'deep_subi_par_match': pd.to_numeric(h['deep_allowed']).mean(),  # moyenne de celles que les adversaires réussissent près de sa surface
        'xG_par_match': pd.to_numeric(h['xG']).mean(),  # moyenne de la qualité des occasions créées
        'xGA_par_match': pd.to_numeric(h['xGA']).mean(),  # moyenne de la qualité des occasions concédées
    })  # fin de la ligne de résumé

style = pd.DataFrame(lignes).set_index('equipe').round(2)  # crée le tableau, utilise le nom d'équipe comme étiquette de ligne, arrondit à 2 décimales
style = style.sort_values('ppda').reset_index()  # trie du pressing le plus intense (PPDA le plus bas) au moins intense
style.index += 1  # fait commencer la numérotation à 1, pour que ce soit un rang lisible


def etiquette_style(ppda):  # petite recette : reçoit un PPDA et renvoie une étiquette
    if ppda < SEUIL_PRESSING_HAUT:  # si le PPDA est sous le seuil du pressing haut
        return 'pressing haut'  # l'équipe presse haut
    elif ppda < SEUIL_BLOC_BAS:  # sinon, si le PPDA est sous le seuil du bloc bas
        return 'pressing median'  # le pressing est médian
    else:  # sinon, le PPDA atteint le seuil du bloc bas
        return 'bloc bas'  # l'équipe joue en bloc bas


style['etiquette'] = style['ppda'].apply(etiquette_style)  # applique la recette à chaque équipe et range le résultat dans une nouvelle colonne

ligne_equipe = style[style['equipe'] == EQUIPE].iloc[0]  # retrouve la ligne de notre équipe grâce à son nom
print(f"{EQUIPE} : {ligne_equipe['etiquette']} (PPDA {ligne_equipe['ppda']}, actions subies près de la surface : {ligne_equipe['deep_subi_par_match']} par match)")  # verdict avec les deux chiffres qui le justifient

print(style['etiquette'].value_counts())  # contrôle : on attend 8 bloc bas, 7 pressing haut et 5 pressing median
style[['equipe', 'ppda', 'deep_subi_par_match', 'etiquette']]  # affiche le tableau des 20 équipes avec les colonnes utiles

AC Milan : bloc bas (PPDA 14.62, actions subies près de la surface : 5.32 par match)
etiquette
bloc bas           8
pressing haut      7
pressing median    5
Name: count, dtype: int64


,equipe,ppda,deep_subi_par_match,etiquette
1,Como,7.71,3.82,pressing haut
2,Roma,8.66,4.53,pressing haut
3,Inter,9.66,3.34,pressing haut
4,Bologna,9.89,4.74,pressing haut
5,Genoa,10.05,6.24,pressing haut
6,Juventus,10.58,4.55,pressing haut
7,Atalanta,10.63,6.21,pressing haut
8,Udinese,11.60,5.32,pressing median
9,Napoli,11.87,3.89,pressing median
10,Lecce,11.93,7.16,pressing median


## 4. Domicile et extérieur : la normale de la ligue

Presque toutes les équipes prennent plus de points à domicile. Dire qu'une équipe est « meilleure à domicile » n'a donc aucun intérêt : il faut la comparer à la normale de la ligue. On calcule pour chaque équipe l'écart entre ses points par match à l'extérieur et à domicile. Un écart positif signifie qu'elle fait mieux dehors. On compare ensuite cet écart à la moyenne des 20 équipes, qui est l'avantage du terrain habituel en Serie A.

**Contrôle attendu** : Milan prend 2,00 points par match à l'extérieur contre 1,68 à domicile, et seules 6 équipes sur 20 font mieux dehors.

In [5]:
tous_matchs['lieu'] = tous_matchs['h_a'].map({'h': 'domicile', 'a': 'exterieur'})  # traduit h et a en mots clairs pour les 20 équipes

ecart_lieu = tous_matchs.groupby(['equipe', 'lieu'])['pts'].mean().unstack()  # calcule les points par match à domicile et à l'extérieur, une ligne par équipe et une colonne par lieu
ecart_lieu['ecart_ext_moins_dom'] = ecart_lieu['exterieur'] - ecart_lieu['domicile']  # écart entre l'extérieur et le domicile, positif si l'équipe fait mieux à l'extérieur
ecart_lieu = ecart_lieu.round(2).sort_values('ecart_ext_moins_dom')  # arrondit à 2 décimales et trie de la plus grande faiblesse à l'extérieur à la plus grande force à l'extérieur

print("Avantage du terrain moyen de la ligue (points par match) :", fr(-ecart_lieu['ecart_ext_moins_dom'].mean()))  # la moyenne de la ligue, signe inversé pour qu'un avantage à domicile soit positif
print("Équipes qui font mieux dehors que chez elles :", int((ecart_lieu['ecart_ext_moins_dom'] > 0).sum()), "sur 20")  # contrôle : 6 équipes selon nos chiffres
ecart_lieu.loc[[EQUIPE]]  # affiche la ligne de l'équipe d'exemple : on attend 2,00 à l'extérieur et 1,68 à domicile

Avantage du terrain moyen de la ligue (points par match) : 0,12
Équipes qui font mieux dehors que chez elles : 6 sur 20


lieu,domicile,exterieur,ecart_ext_moins_dom
equipe,,,
AC Milan,1.68,2.0,0.32


## 5. Les tirs de toute la ligue

Pour comparer une équipe à la ligue sur les coups de pied arrêtés et sur l'origine des buts, il faut les tirs des 380 matchs. Le téléchargement dure environ 7 minutes (une pause d'une seconde entre chaque demande, pour rester poli avec le site). Il n'est fait qu'une fois : le résultat est sauvegardé dans un fichier, qui est relu aux lancements suivants.

La colonne `equipe_but` corrige les buts contre son camp : le but est attribué à l'équipe qui en profite.

**Contrôles attendus** : 9 373 tirs, 380 matchs couverts, 0 échec, et 940 tirs dans les matchs de Milan.

In [6]:
FICHIER_TIRS_LIGUE = f'data/raw/tirs_serie_a_{SAISON}-{SAISON + 1}.csv'  # fichier où seront sauvegardés les tirs des 20 équipes, pour ne les demander qu'une seule fois

if os.path.exists(FICHIER_TIRS_LIGUE):  # si le fichier existe déjà, on ne redemande rien au site
    tirs_ligue = pd.read_csv(FICHIER_TIRS_LIGUE)  # relit le fichier sauvegardé
    print("Fichier existant relu, aucune demande envoyée :", tirs_ligue.shape)  # affiche la taille du tableau relu
else:  # sinon, il faut collecter les tirs de tous les matchs de la saison
    ids_matchs = set()  # ensemble vide qui accueillera les identifiants de matchs, sans doublon
    for team_id, info in league_table.items():  # parcourt les 20 équipes
        try:  # essaie de récupérer les matchs de cette équipe
            liste = understat.team(team=info['title'].replace(' ', '_')).get_match_data(season=str(SAISON))  # liste des matchs de l'équipe avec leur identifiant
            for m in liste:  # parcourt chaque match de la liste
                ids_matchs.add(m['id'])  # ajoute l'identifiant, un match joué par deux équipes n'est gardé qu'une fois
        except Exception as erreur:  # si la demande échoue pour cette équipe
            print("Échec pour", info['title'], ":", str(erreur)[:100])  # affiche l'équipe en cause et le début du message d'erreur
        time.sleep(1)  # attend 1 seconde avant l'équipe suivante

    print("Matchs distincts trouvés :", len(ids_matchs))  # contrôle : on attend 380 pour une saison complète

    tous_tirs_ligue = []  # liste vide qui accueillera les tirs de tous les matchs
    echecs = []  # liste vide qui notera les matchs dont la collecte a échoué
    for n, id_match in enumerate(sorted(ids_matchs), 1):  # parcourt les matchs un par un, n est le compteur
        try:  # essaie de récupérer les tirs de ce match
            donnees = understat.match(match=id_match).get_shot_data()  # demande les tirs du match, pour les deux équipes
            tous_tirs_ligue.extend(donnees['h'] + donnees['a'])  # ajoute les tirs de l'équipe à domicile et de celle à l'extérieur
        except Exception as erreur:  # si la demande échoue pour ce match
            echecs.append((id_match, str(erreur)[:80]))  # note le match et le début du message d'erreur
        time.sleep(1)  # attend 1 seconde avant le match suivant, pour rester poli avec le site
        if n % 50 == 0:  # toutes les 50 demandes
            print(n, "matchs traités sur", len(ids_matchs))  # affiche l'avancement

    tirs_ligue = pd.DataFrame(tous_tirs_ligue)  # transforme la liste en tableau, un tir par ligne
    for col in ['minute', 'X', 'Y', 'xG']:  # colonnes qui doivent être des nombres
        tirs_ligue[col] = pd.to_numeric(tirs_ligue[col])  # convertit en nombres

    tirs_ligue['equipe'] = tirs_ligue['h_team'].where(tirs_ligue['h_a'] == 'h', tirs_ligue['a_team'])  # équipe qui tire : l'équipe à domicile si le tir vient du côté h, sinon celle à l'extérieur
    tirs_ligue['adversaire'] = tirs_ligue['a_team'].where(tirs_ligue['h_a'] == 'h', tirs_ligue['h_team'])  # équipe qui subit le tir : l'autre
    tirs_ligue['equipe_but'] = tirs_ligue['equipe']  # équipe qui bénéficie du but, identique à l'équipe qui tire sauf but contre son camp
    masque = tirs_ligue['result'] == 'OwnGoal'  # repère les buts contre son camp
    tirs_ligue.loc[masque, 'equipe_but'] = tirs_ligue.loc[masque, 'adversaire']  # un but contre son camp profite à l'autre équipe

    tirs_ligue.to_csv(FICHIER_TIRS_LIGUE, index=False)  # sauvegarde les tirs de la ligue pour ne pas avoir à tout redemander
    print("Matchs en échec :", len(echecs), echecs[:5])  # contrôle : on attend 0 échec, sinon on relance en listant les matchs concernés

print(tirs_ligue.shape)  # affiche (nombre de tirs, nombre de colonnes)
print("Matchs couverts :", tirs_ligue['match_id'].nunique())  # contrôle : on attend 380
print("Tirs des matchs de l'équipe :", ((tirs_ligue['equipe'] == EQUIPE) | (tirs_ligue['adversaire'] == EQUIPE)).sum())  # contrôle : on attend 940 pour l'AC Milan

Fichier existant relu, aucune demande envoyée : (9373, 23)
(9373, 23)
Matchs couverts : 380
Tirs des matchs de l'équipe : 940


## 6. Collecte Sofascore : pourquoi et comment

Understat donne le style, les tirs et les buts. Pour les duels, les interceptions, les compositions et les statistiques de chaque joueur, il faut Sofascore. Le site ne fournit pas d'export officiel, et un script classique est bloqué. On passe donc par un vrai navigateur (Playwright) qui demande les données comme le fait le site lui même, avec une pause de 1 à 2 secondes entre chaque demande.

Règles de la collecte :
1. Uniquement la saison 2025/26 (identifiant Sofascore 76457) et la Serie A (identifiant 23).
2. Chaque réponse est sauvegardée dans un fichier. Si la collecte s'arrête, on la relance : elle ne télécharge que ce qui manque.
3. Une fenêtre de navigateur s'ouvre pendant le téléchargement. Il ne faut pas la fermer.

Le script de collecte est écrit dans un fichier à part (`sofascore_collecte.py`), car il doit tourner en dehors du notebook.

In [7]:
!pip install playwright
!python -m playwright install chromium

In [8]:
%%writefile sofascore_collecte.py
import sys  # pour lire les informations données au lancement du script
import os  # pour créer le dossier de sauvegarde et tester l'existence des fichiers
import re  # pour fabriquer des noms de fichiers à partir des adresses
import random  # pour varier légèrement les pauses entre les demandes
from playwright.sync_api import sync_playwright  # pour piloter un vrai navigateur

dossier = sys.argv[1]  # dossier où seront sauvegardées les données, donné au lancement
chemins = []  # liste de toutes les adresses de données à demander, sans le nom du site
for element in sys.argv[2:]:  # parcourt les informations données après le dossier
    if element.endswith('.txt'):  # si c'est un fichier texte, il contient une adresse par ligne
        with open(element, encoding='utf-8') as f:  # ouvre ce fichier
            chemins.extend([ligne.strip() for ligne in f if ligne.strip()])  # ajoute chaque ligne non vide à la liste
    else:  # sinon, c'est directement une adresse
        chemins.append(element)  # l'ajoute à la liste
os.makedirs(dossier, exist_ok=True)  # crée le dossier s'il n'existe pas encore


def nom_fichier(chemin):  # petite recette : fabrique le nom du fichier de sauvegarde d'une adresse
    return re.sub('[^a-zA-Z0-9]+', '_', chemin).strip('_') + '.json'  # remplace tout caractère spécial par un tiret bas


a_faire = [c for c in chemins if not os.path.exists(os.path.join(dossier, nom_fichier(c)))]  # garde seulement les adresses dont le fichier n'existe pas encore, ce qui permet de reprendre après une coupure
print("Adresses demandées :", len(chemins), "| déjà sauvegardées :", len(chemins) - len(a_faire), "| à télécharger :", len(a_faire))  # affiche le bilan avant de commencer

echecs = []  # liste des adresses qui n'ont pas pu être téléchargées

if a_faire:  # s'il y a quelque chose à télécharger
    with sync_playwright() as p:  # démarre le pilote de navigateur
        navigateur = p.chromium.launch(headless=False)  # ouvre un vrai navigateur visible
        page = navigateur.new_page()  # ouvre un nouvel onglet
        page.goto('https://www.sofascore.com/fr/', wait_until='domcontentloaded', timeout=60000)  # ouvre la page d'accueil de Sofascore, pour être dans les conditions d'une vraie visite
        page.wait_for_timeout(8000)  # attend 8 secondes pour laisser la page s'installer
        for n, chemin in enumerate(a_faire, 1):  # parcourt les adresses à télécharger, n est le compteur
            try:  # essaie de télécharger cette adresse
                resultat = page.evaluate("async (chemin) => { const r = await fetch(chemin, {credentials: 'include'}); const t = await r.text(); return {statut: r.status, texte: t}; }", chemin)  # demande la donnée depuis la page elle-même, comme le fait le site
            except Exception as erreur:  # si la demande plante
                echecs.append((chemin, str(erreur)[:80]))  # note l'adresse et le début du message d'erreur
                continue  # passe à l'adresse suivante
            if resultat['statut'] == 200:  # si la demande a réussi
                with open(os.path.join(dossier, nom_fichier(chemin)), 'w', encoding='utf-8') as f:  # ouvre le fichier de sauvegarde
                    f.write(resultat['texte'])  # sauvegarde la donnée reçue
            else:  # sinon, la demande a été refusée ou l'adresse n'existe pas
                echecs.append((chemin, resultat['statut']))  # note l'adresse et le code de réponse
            if n % 10 == 0:  # toutes les 10 adresses
                print(n, "sur", len(a_faire), "traitées")  # affiche l'avancement
            page.wait_for_timeout(random.randint(1000, 2000))  # attend entre 1 et 2 secondes avant la demande suivante, pour rester poli avec le site
        navigateur.close()  # ferme le navigateur

print("Échecs :", len(echecs))  # affiche le nombre d'échecs, on en attend 0
for e in echecs[:10]:  # parcourt les dix premiers échecs
    print(e)  # affiche l'adresse concernée et la cause

Overwriting sofascore_collecte.py


## 7. Les 20 équipes et les 380 matchs côté Sofascore

Trois téléchargements se suivent, chacun préparant le suivant :
1. le classement de la saison, qui donne les 20 équipes et leur numéro Sofascore ;
2. quatre pages de matchs récents par équipe, soit 80 pages, d'où l'on tire les 380 matchs de Serie A terminés (chaque match apparaît chez les deux équipes, il faut retirer les doublons) ;
3. pour chaque match, ses statistiques d'équipe et sa composition avec les statistiques de chaque joueur, soit 760 fichiers.

**Contrôles attendus** : 20 équipes aux numéros distincts, 80 adresses de pages, puis 380 matchs terminés avec 38 matchs pour chaque équipe, puis 760 adresses de matchs. Le dernier téléchargement dure environ 25 minutes la première fois.

In [9]:
DOSSIER_LIGUE = 'data/raw/sofascore_ligue'  # dossier où sont sauvegardés les fichiers Sofascore de toute la ligue
SOFA_ID_SERIE_A = 23  # numéro de la Serie A chez Sofascore
SOFA_ID_SAISON = 76457  # numéro de la saison 2025/26 chez Sofascore
os.makedirs(DOSSIER_LIGUE, exist_ok=True)  # crée le dossier s'il n'existe pas encore


def fouiller(objet, condition, trouves):  # petite recette qui fouille tout le contenu d'un fichier, même imbriqué, et garde les éléments qui respectent la condition
    if isinstance(objet, dict):  # si l'objet est un dictionnaire
        if condition(objet):  # s'il respecte la condition demandée
            trouves.append(objet)  # on le garde
        for valeur in objet.values():  # parcourt toutes ses valeurs
            fouiller(valeur, condition, trouves)  # fouille chacune
    elif isinstance(objet, list):  # si l'objet est une liste
        for element in objet:  # parcourt ses éléments
            fouiller(element, condition, trouves)  # fouille chacun

In [10]:
!python sofascore_collecte.py "data/raw/sofascore_ligue" "/api/v1/unique-tournament/23/season/76457/standings/total"

Adresses demandées : 1 | déjà sauvegardées : 1 | à télécharger : 0
Échecs : 0


In [11]:
with open(os.path.join(DOSSIER_LIGUE, f"api_v1_unique_tournament_{SOFA_ID_SERIE_A}_season_{SOFA_ID_SAISON}_standings_total.json"), encoding="utf-8") as f:  # ouvre le classement sauvegardé de la saison 2025/26
    classement_brut = json.load(f)  # lit son contenu

lignes_classement = []  # liste des lignes de classement trouvées
fouiller(classement_brut, lambda d: isinstance(d.get('team'), dict) and 'points' in d and 'position' in d, lignes_classement)  # garde tout ce qui ressemble à une ligne de classement : une équipe, des points, une position

equipes_sofa = pd.DataFrame([{  # construit le tableau des équipes de la Serie A
    'id_equipe': l['team'].get('id'),  # numéro de l'équipe chez Sofascore
    'equipe': l['team'].get('name'),  # nom de l'équipe chez Sofascore
    'rang': l.get('position'),  # place au classement final
    'points': l.get('points'),  # points de la saison
} for l in lignes_classement]).sort_values('rang').reset_index(drop=True)  # trie par place au classement

print("Équipes trouvées :", len(equipes_sofa), "| numéros distincts :", equipes_sofa['id_equipe'].nunique())  # contrôle : on attend 20 équipes et 20 numéros distincts
equipes_sofa.to_csv('data/processed/sofascore_equipes.csv', index=False)  # sauvegarde la liste des équipes

with open(f'{DOSSIER_LIGUE}/chemins_pages.txt', 'w', encoding='utf-8') as f:  # ouvre le fichier qui listera les adresses à télécharger
    for id_equipe in equipes_sofa['id_equipe']:  # parcourt les 20 équipes
        for page in range(4):  # parcourt les quatre pages de matchs récents de chaque équipe
            f.write(f"/api/v1/team/{int(id_equipe)}/events/last/{page}\n")  # écrit l'adresse de cette page de matchs
print("Adresses écrites :", 4 * len(equipes_sofa), "| attendu : 80")  # contrôle : on attend 80
equipes_sofa  # affiche les 20 équipes avec leur numéro

Équipes trouvées : 20 | numéros distincts : 20
Adresses écrites : 80 | attendu : 80


,id_equipe,equipe,rang,points
0,2697,Inter,1,87
1,2714,SSC Napoli,2,76
2,2702,AS Roma,3,73
3,2704,Como,4,71
4,2692,AC Milan,5,70
5,2687,Juventus,6,69
6,2686,Atalanta,7,59
7,2685,Bologna,8,56
8,2699,Lazio,9,54
9,2695,Udinese,10,50


In [12]:
!python sofascore_collecte.py "data/raw/sofascore_ligue" "data/raw/sofascore_ligue/chemins_pages.txt"

Adresses demandées : 80 | déjà sauvegardées : 80 | à télécharger : 0
Échecs : 0


In [13]:
import glob  # pour lister les fichiers déjà téléchargés

fichiers_pages = sorted(glob.glob(f'{DOSSIER_LIGUE}/*events_last*.json'))  # retrouve les 80 pages de matchs sauvegardées
print("Fichiers de pages trouvés :", len(fichiers_pages), "| attendu : 80")  # contrôle : on attend 80

evenements = []  # liste de tous les matchs trouvés, avec doublons (un match apparaît chez les deux équipes)
for chemin in fichiers_pages:  # parcourt chaque page sauvegardée
    with open(chemin, encoding='utf-8') as f:  # ouvre le fichier de la page
        page_brute = json.load(f)  # lit son contenu
    fouiller(page_brute, lambda d: 'homeTeam' in d and 'awayTeam' in d and 'status' in d and 'id' in d, evenements)  # garde tout ce qui ressemble à un match

lignes = []  # lignes du tableau final
for e in evenements:  # parcourt chaque match trouvé
    tournoi = (e.get('tournament') or {}).get('uniqueTournament') or {}  # compétition du match (Serie A, coupe, etc.)
    lignes.append({  # une ligne par match
        'id_match': e['id'],  # numéro du match chez Sofascore
        'id_competition': tournoi.get('id'),  # 23 = Serie A
        'id_saison': (e.get('season') or {}).get('id'),  # 76457 = saison 2025/26
        'etat': (e.get('status') or {}).get('type'),  # finished, postponed, etc.
        'date': pd.to_datetime(e.get('startTimestamp'), unit='s'),  # date du match
        'tour': (e.get('roundInfo') or {}).get('round'),  # numéro de journée
        'id_domicile': e['homeTeam'].get('id'),  # équipe à domicile
        'domicile': e['homeTeam'].get('name'),  # nom de l'équipe à domicile
        'id_exterieur': e['awayTeam'].get('id'),  # équipe à l'extérieur
        'exterieur': e['awayTeam'].get('name'),  # nom de l'équipe à l'extérieur
        'buts_dom': (e.get('homeScore') or {}).get('current'),  # buts de l'équipe à domicile
        'buts_ext': (e.get('awayScore') or {}).get('current'),  # buts de l'équipe à l'extérieur
    })
tous = pd.DataFrame(lignes)  # tableau de tous les matchs trouvés
print("Matchs bruts :", len(tous), "| distincts :", tous['id_match'].nunique())  # contrôle des doublons

matchs_ligue = (tous[(tous['id_competition'] == SOFA_ID_SERIE_A) & (tous['id_saison'] == SOFA_ID_SAISON) & (tous['etat'] == 'finished')]  # garde Serie A, saison 2025/26, matchs terminés
                .drop_duplicates('id_match').sort_values(['date', 'id_match']).reset_index(drop=True))  # supprime les doublons et trie par date
print("Matchs Serie A terminés :", len(matchs_ligue), "| attendu : 380")  # contrôle principal
print("Matchs par équipe (attendu 38 partout) :")  # contrôle par équipe
print(pd.concat([matchs_ligue['domicile'], matchs_ligue['exterieur']]).value_counts().to_string())  # compte les matchs de chaque équipe

matchs_ligue.to_csv('data/processed/sofascore_matchs_ligue.csv', index=False)  # sauvegarde la liste des matchs

with open(f'{DOSSIER_LIGUE}/chemins_matchs.txt', 'w', encoding='utf-8') as f:  # fichier des adresses à télécharger
    for id_match in matchs_ligue['id_match']:  # parcourt les matchs
        f.write(f"/api/v1/event/{int(id_match)}/statistics\n")  # statistiques d'équipe du match
        f.write(f"/api/v1/event/{int(id_match)}/lineups\n")  # compositions et statistiques joueurs du match
print("Adresses écrites :", 2 * len(matchs_ligue), "| attendu : 760")  # contrôle

Fichiers de pages trouvés : 80 | attendu : 80
Matchs bruts : 2400 | distincts : 1643
Matchs Serie A terminés : 380 | attendu : 380
Matchs par équipe (attendu 38 partout) :
Genoa            38
Sassuolo         38
AS Roma          38
AC Milan         38
Como             38
Cagliari         38
Juventus         38
Atalanta         38
Udinese          38
Inter            38
Cremonese        38
Lecce            38
Parma            38
Bologna          38
SSC Napoli       38
Pisa             38
Torino           38
Lazio            38
Fiorentina       38
Hellas Verona    38
Adresses écrites : 760 | attendu : 760


In [14]:
!python sofascore_collecte.py "data/raw/sofascore_ligue" "data/raw/sofascore_ligue/chemins_matchs.txt"

Adresses demandées : 760 | déjà sauvegardées : 760 | à télécharger : 0
Échecs : 0


## 8. Nettoyage : statistiques de joueurs

Dans les fichiers de composition, chaque joueur a une ligne par match avec ses minutes et ses statistiques. Deux précautions :
1. Le camp joué (domicile ou extérieur) est lu dans le fichier, pas déduit du club d'origine du joueur, car un joueur transféré en cours de saison a joué pour deux équipes.
2. Un joueur sans minutes est retiré : il n'a pas joué, donc ses statistiques sont vides et non nulles.

**Contrôles attendus** : 0 fichier manquant, 380 matchs, 586 joueurs distincts, 760 camps sur 760 avec 11 titulaires, et des minutes par camp autour de 990 (le minimum de 890 et le maximum de 1018 s'expliquent par des matchs raccourcis par une expulsion ou allongés par des arrêts de jeu).

In [15]:
lignes_joueur = []  # statistiques joueurs, une ligne par match et par joueur
manquants = []  # matchs dont un fichier est absent

for m in matchs_ligue.itertuples():  # parcourt les 380 matchs
    f_stat = os.path.join(DOSSIER_LIGUE, f"api_v1_event_{m.id_match}_statistics.json")  # fichier des statistiques d'équipe de ce match
    f_compo = os.path.join(DOSSIER_LIGUE, f"api_v1_event_{m.id_match}_lineups.json")  # fichier des compositions de ce match
    if not (os.path.exists(f_stat) and os.path.exists(f_compo)):  # si l'un des deux fichiers manque
        manquants.append(m.id_match)  # on note le match pour le signaler
        continue  # et on passe au suivant

    with open(f_compo, encoding='utf-8') as f:  # ouvre les compositions
        brut_compo = json.load(f)  # lit le contenu
    for cote in ('home', 'away'):  # parcourt les deux camps
        bloc = brut_compo.get(cote) or {}  # données de ce camp
        for p in bloc.get('players', []):  # parcourt les joueurs du groupe
            st = p.get('statistics') or {}  # statistiques du joueur dans ce match
            ligne = {  # base de la ligne du joueur
                'id_match': m.id_match,  # numéro du match
                'cote': cote,  # camp joué (et non le club d'origine)
                'id_equipe': m.id_domicile if cote == 'home' else m.id_exterieur,  # équipe réelle pour ce match
                'formation': bloc.get('formation'),  # schéma utilisé
                'id_joueur': (p.get('player') or {}).get('id'),  # numéro du joueur
                'joueur': (p.get('player') or {}).get('name'),  # nom du joueur
                'poste': p.get('position') or (p.get('player') or {}).get('position'),  # poste
                'remplacant': p.get('substitute'),  # True si parti du banc
            }
            ligne.update({k: v for k, v in st.items() if isinstance(v, (int, float))})  # ajoute toutes les statistiques numériques
            lignes_joueur.append(ligne)  # enregistre la ligne

joueurs_ligue = pd.DataFrame(lignes_joueur)  # tableau des stats joueurs de toute la ligue
joueurs_ligue = joueurs_ligue[joueurs_ligue['minutesPlayed'].notna()].copy()  # retire ceux qui n'ont pas joué (pas de minutes)

print("Matchs avec fichiers manquants :", len(manquants), "| attendu : 0")  # contrôle : on attend 0
print("Matchs dans joueurs :", joueurs_ligue['id_match'].nunique(), "| attendu : 380")  # contrôle : on attend 380
print("Joueurs distincts :", joueurs_ligue['id_joueur'].nunique(), "| lignes joueur et match :", len(joueurs_ligue))  # volume joueurs, on attend 586 joueurs distincts
titulaires = joueurs_ligue[~joueurs_ligue['remplacant']].groupby(['id_match', 'cote']).size()  # nombre de titulaires par match et camp
print("Camps avec 11 titulaires :", int((titulaires == 11).sum()), "sur", len(titulaires), "(attendu 760 sur 760)")  # contrôle de cohérence des compositions
minutes_equipe = joueurs_ligue.groupby(['id_match', 'cote'])['minutesPlayed'].sum()  # total de minutes jouées par camp
print("Minutes par camp : min", minutes_equipe.min(), "| médiane", minutes_equipe.median(), "| max", minutes_equipe.max())  # on attend environ 990 pour un match de 90 minutes

joueurs_ligue.to_csv('data/processed/sofascore_joueurs_ligue.csv', index=False)  # sauvegarde les stats joueurs

Matchs avec fichiers manquants : 0 | attendu : 0
Matchs dans joueurs : 380 | attendu : 380
Joueurs distincts : 586 | lignes joueur et match : 11928
Camps avec 11 titulaires : 760 sur 760 (attendu 760 sur 760)
Minutes par camp : min 890.0 | médiane 990.0 | max 1018.0


## 9. Nettoyage : statistiques d'équipe

Les statistiques d'équipe de Sofascore ont deux pièges :
1. **Des statistiques en double** : certaines apparaissent dans deux groupes différents (par exemple des duels dans « duels » et dans « défense »). On vérifie que les copies sont identiques, puis on n'en garde qu'une.
2. **Des pourcentages qui sont en réalité des comptes** : pour les duels, les tacles ou les dribbles, la valeur est le nombre de réussites, et le nombre de tentatives est donné à part (`total`). On garde les deux, ce qui permet de recalculer un vrai taux de réussite.

Le résultat est un tableau large : une ligne par équipe et par match, une colonne par statistique.

**Contrôles attendus** : 0 cas où les copies diffèrent, 760 lignes au maximum par statistique (au lieu de 1520), un tableau de 760 lignes et 20 équipes.

In [16]:
lignes_equipe = []  # liste des statistiques d'équipe, avec le total et le groupe d'origine
for m in matchs_ligue.itertuples():  # parcourt les 380 matchs
    with open(os.path.join(DOSSIER_LIGUE, f"api_v1_event_{m.id_match}_statistics.json"), encoding='utf-8') as f:  # ouvre les statistiques d'équipe du match
        brut_stat = json.load(f)  # lit le contenu
    for periode in brut_stat.get('statistics', []):  # parcourt les périodes
        if periode.get('period') != 'ALL':  # garde seulement le match complet
            continue  # ignore les mi-temps
        for groupe in periode.get('groups', []):  # parcourt les groupes
            for s in groupe.get('statisticsItems', []):  # parcourt les statistiques
                for cote in ('home', 'away'):  # une valeur pour chaque camp
                    lignes_equipe.append({  # une ligne par match, camp et statistique
                        'id_match': m.id_match,  # numéro du match
                        'cote': cote,  # camp : domicile ou extérieur
                        'id_equipe': m.id_domicile if cote == 'home' else m.id_exterieur,  # équipe correspondante
                        'groupe': groupe.get('groupName'),  # groupe d'origine dans Sofascore
                        'stat': s.get('key'),  # nom technique de la statistique
                        'valeur': s.get(cote + 'Value'),  # valeur (nombre de réussites pour les duels, tacles et dribbles)
                        'total': s.get(cote + 'Total'),  # total des tentatives quand il existe
                    })
stats_brut = pd.DataFrame(lignes_equipe)  # tableau complet avec les doublons

doublons = stats_brut[stats_brut.duplicated(['id_match', 'cote', 'stat'], keep=False)]  # lignes présentes plusieurs fois pour un même match, camp et statistique
ecarts = doublons.groupby(['id_match', 'cote', 'stat'])['valeur'].nunique()  # nombre de valeurs différentes parmi les copies
print("Statistiques en double :", sorted(doublons['stat'].unique()))  # liste des statistiques concernées
print("Cas où les copies ont des valeurs différentes :", int((ecarts > 1).sum()), "(attendu 0)")  # contrôle : les copies doivent être identiques

stats_equipe_ligue = stats_brut.drop_duplicates(['id_match', 'cote', 'stat']).copy()  # garde une seule copie de chaque statistique
stats_equipe_ligue['valeur'] = pd.to_numeric(stats_equipe_ligue['valeur'], errors='coerce')  # valeurs en nombres
stats_equipe_ligue['total'] = pd.to_numeric(stats_equipe_ligue['total'], errors='coerce')  # totaux en nombres
print("Statistiques distinctes :", stats_equipe_ligue['stat'].nunique(), "| attendu : 50")  # nombre de types de statistiques
print("Couverture max par statistique :", stats_equipe_ligue.groupby('stat').size().max(), "(attendu 760)")  # contrôle : plus aucune statistique à 1520

camps = (stats_equipe_ligue.pivot_table(index=['id_match', 'cote', 'id_equipe'], columns='stat', values='valeur')  # un tableau large : une ligne par match et camp, une colonne par statistique
         .reset_index())  # remet les clés en colonnes
totaux = (stats_equipe_ligue.dropna(subset=['total']).pivot_table(index=['id_match', 'cote'], columns='stat', values='total')  # totaux des statistiques en réussites sur tentatives
          .add_suffix('_total').reset_index())  # suffixe _total pour les distinguer
camps = camps.merge(totaux, on=['id_match', 'cote'], how='left')  # joint valeurs et totaux
print("Tableau des camps :", camps.shape[0], "lignes (attendu 760) |", camps['id_equipe'].nunique(), "équipes (attendu 20)")  # contrôle de forme et nombre d'équipes

stats_equipe_ligue.to_csv('data/processed/sofascore_stats_equipe_ligue.csv', index=False)  # sauvegarde la version corrigée
camps.to_csv('data/processed/sofascore_camps_ligue.csv', index=False)  # sauvegarde le tableau large
camps.head(3)  # aperçu

Statistiques en double : ['expectedGoals', 'goalkeeperSaves', 'totalShotsOnGoal', 'totalTackle']
Cas où les copies ont des valeurs différentes : 0 (attendu 0)
Statistiques distinctes : 50 | attendu : 50
Couverture max par statistique : 760 (attendu 760)
Tableau des camps : 760 lignes (attendu 760) | 20 équipes (attendu 20)


stat,id_match,cote,id_equipe,accurateCross,accurateLongBalls,accuratePasses,accurateThroughBall,aerialDuelsPercentage,ballPossession,ballRecovery,...,touchesInOppBox,wonTacklePercent,yellowCards,accurateCross_total,accurateLongBalls_total,aerialDuelsPercentage_total,dribblesPercentage_total,finalThirdPhaseStatistic_total,groundDuelsPercentage_total,wonTacklePercent_total
0,13980058,away,2685,5.0,22.0,396.0,0.0,10.0,55.0,42.0,...,20.0,3.0,1.0,16.0,47.0,18.0,22.0,132.0,73.0,8.0
1,13980058,home,2686,5.0,17.0,335.0,1.0,8.0,45.0,51.0,...,18.0,18.0,1.0,10.0,44.0,18.0,11.0,105.0,73.0,26.0
2,13980059,away,2704,6.0,22.0,442.0,NaN,17.0,64.0,47.0,...,29.0,5.0,1.0,23.0,40.0,32.0,22.0,144.0,85.0,13.0


## 10. Relier Understat et Sofascore : les équipes

Les deux sites n'écrivent pas toujours les noms de la même façon. Pour croiser les données, il faut une table de correspondance. On procède en trois essais, du plus sûr au moins sûr :
1. nom identique une fois les accents, les points et les tirets retirés ;
2. un nom contenu dans l'autre, s'il n'y a qu'un seul candidat ;
3. à défaut, le nom le plus ressemblant, marqué « à vérifier ».

On ne fait jamais confiance à une correspondance approximative sans la regarder : le tableau affiché est à relire.

**Contrôles attendus** : 0 nom Understat utilisé deux fois, 0 nom Understat oublié, et les 20 correspondances justes à l'œil.

In [17]:
import unicodedata  # pour retirer les accents des noms
import difflib  # pour chercher le nom le plus ressemblant


def normaliser(texte):  # petite recette : met un nom sous une forme comparable
    texte = unicodedata.normalize('NFKD', str(texte)).encode('ascii', 'ignore').decode()  # retire les accents (ex : š devient s)
    return texte.lower().replace('.', '').replace('-', ' ').strip()  # minuscules, sans points ni tirets


noms_us = sorted(tirs_ligue['equipe'].unique())  # noms des 20 équipes chez Understat
noms_so = list(equipes_sofa['equipe'])  # noms des 20 équipes chez Sofascore
norm_us = {normaliser(n): n for n in noms_us}  # nom normalisé vers nom d'origine, côté Understat

lignes = []  # tableau de correspondance
for id_eq, nom_so in zip(equipes_sofa['id_equipe'], noms_so):  # parcourt les équipes Sofascore
    n = normaliser(nom_so)  # forme comparable du nom Sofascore
    if n in norm_us:  # nom identique après normalisation
        choix, methode = norm_us[n], 'identique'  # correspondance exacte
    else:  # sinon on cherche si un nom est contenu dans l'autre, puis le plus proche
        contenus = [o for k, o in norm_us.items() if k in n or n in k]  # noms qui contiennent l'autre ou en font partie
        if len(contenus) == 1:  # un seul candidat
            choix, methode = contenus[0], 'contenu'  # correspondance par inclusion
        else:  # aucun ou plusieurs candidats
            proche = difflib.get_close_matches(n, list(norm_us.keys()), n=1, cutoff=0.0)  # nom Understat le plus ressemblant
            choix, methode = (norm_us[proche[0]] if proche else None), 'proche (à vérifier)'  # correspondance approximative à contrôler
    lignes.append({'id_equipe': id_eq, 'sofascore': nom_so, 'understat': choix, 'methode': methode})  # une ligne par équipe

correspondance_equipes = pd.DataFrame(lignes)  # tableau de correspondance
print("Noms Understat utilisés plusieurs fois :", correspondance_equipes['understat'].duplicated().sum(), "(attendu 0)")  # contrôle : une équipe Understat ne peut correspondre qu'à une seule équipe Sofascore
print("Noms Understat non utilisés :", sorted(set(noms_us) - set(correspondance_equipes['understat'])), "(attendu [])")  # contrôle : les 20 doivent être pris

correspondance_equipes.to_csv('data/processed/correspondance_equipes.csv', index=False)  # sauvegarde la correspondance des équipes
us_vers_so = dict(zip(correspondance_equipes['understat'], correspondance_equipes['id_equipe']))  # nom Understat vers numéro Sofascore de l'équipe
correspondance_equipes  # affiche la correspondance à valider

Noms Understat utilisés plusieurs fois : 0 (attendu 0)
Noms Understat non utilisés : [] (attendu [])


,id_equipe,sofascore,understat,methode
0,2697,Inter,Inter,identique
1,2714,SSC Napoli,Napoli,contenu
2,2702,AS Roma,Roma,contenu
3,2704,Como,Como,identique
4,2692,AC Milan,AC Milan,identique
5,2687,Juventus,Juventus,identique
6,2686,Atalanta,Atalanta,identique
7,2685,Bologna,Bologna,identique
8,2699,Lazio,Lazio,identique
9,2695,Udinese,Udinese,identique


## 11. Relier Understat et Sofascore : les joueurs

Même principe pour les joueurs, avec une règle de prudence : on cherche un joueur uniquement dans l'effectif Sofascore de **sa propre équipe**, ce qui évite de confondre deux joueurs homonymes de clubs différents. Les essais, du plus sûr au moins sûr :
1. nom identique après retrait des accents ;
2. même nom de famille, s'il n'y a qu'un seul candidat dans l'effectif ;
3. nom ressemblant à au moins 80 %.

Les noms Understat contiennent parfois des codes web (par exemple `&#039;` à la place d'une apostrophe). On les convertit avant de comparer.

Les cas qui échappent à ces règles sont réglés à la main, un par un, dans la cellule suivante. Ce sont des cas où le nom Understat est très différent du nom Sofascore : par exemple « Franck Zambo » côté Understat est « Frank Anguissa » côté Sofascore.

In [18]:
import html  # pour transformer les codes web comme &#039; en vrais caractères

tirs_ligue['player'] = tirs_ligue['player'].map(lambda x: html.unescape(x) if isinstance(x, str) else x)  # corrige les noms des tireurs
tirs_ligue['player_assisted'] = tirs_ligue['player_assisted'].map(lambda x: html.unescape(x) if isinstance(x, str) else x)  # corrige les noms des passeurs
print("Noms encore avec un code web dans les tirs :", int(tirs_ligue['player'].astype(str).str.contains('&#').sum()), "(attendu 0)")  # contrôle
tirs_ligue.to_csv('data/processed/tirs_ligue_propre.csv', index=False)  # sauvegarde les tirs avec les noms corrigés

effectifs = (joueurs_ligue.groupby(['id_equipe', 'id_joueur'])['joueur'].first().reset_index())  # joueurs Sofascore de chaque équipe, avec leur nom
joueurs_us = pd.concat([  # joueurs Understat : tireurs, et passeurs vers un tir
    tirs_ligue[tirs_ligue['result'] != 'OwnGoal'][['equipe', 'player']].rename(columns={'player': 'joueur_us'}),  # tireurs avec leur équipe
    tirs_ligue[tirs_ligue['player_assisted'].notna() & (tirs_ligue['player_assisted'] != '')][['equipe', 'player_assisted']].rename(columns={'player_assisted': 'joueur_us'}),  # passeurs avec l'équipe qui tire
]).drop_duplicates().reset_index(drop=True)  # une ligne par équipe et joueur

lignes = []  # tableau de correspondance joueurs
for l in joueurs_us.itertuples():  # parcourt chaque joueur Understat
    eff = effectifs[effectifs['id_equipe'] == us_vers_so[l.equipe]].copy()  # effectif Sofascore de son équipe
    eff['norm'] = eff['joueur'].map(normaliser)  # noms Sofascore normalisés
    n = normaliser(l.joueur_us)  # nom Understat normalisé
    exact = eff[eff['norm'] == n]  # nom identique
    if len(exact) == 1:  # une seule correspondance exacte
        choix, methode = exact.iloc[0], 'identique'  # accepté
    else:  # sinon on tente le nom de famille
        famille = n.split(' ')[-1]  # dernier mot du nom Understat
        cand = eff[eff['norm'].map(lambda x: famille in x.split(' '))]  # joueurs de l'effectif ayant ce mot dans leur nom
        if len(cand) == 1:  # un seul candidat
            choix, methode = cand.iloc[0], 'nom de famille (à vérifier)'  # accepté mais à contrôler
        else:  # aucun ou plusieurs candidats : on prend le plus ressemblant
            scores = eff['norm'].map(lambda x: max(difflib.SequenceMatcher(None, n, x).ratio(), difflib.SequenceMatcher(None, famille, x.split(' ')[-1]).ratio()))  # ressemblance du nom complet ou du nom de famille, la meilleure des deux
            if len(scores) and scores.max() >= 0.8:  # assez ressemblant
                choix, methode = eff.loc[scores.idxmax()], 'proche (à vérifier)'  # accepté mais à contrôler
            else:  # rien de convaincant
                choix, methode = None, 'aucun'  # non trouvé
    lignes.append({'equipe_us': l.equipe, 'joueur_us': l.joueur_us, 'id_joueur': None if choix is None else choix['id_joueur'], 'joueur_sofa': None if choix is None else choix['joueur'], 'methode': methode})  # une ligne par joueur

correspondance_joueurs = pd.DataFrame(lignes)  # tableau de correspondance
print(correspondance_joueurs['methode'].value_counts().to_string())  # nombre de joueurs par méthode
print("Même joueur Sofascore associé à plusieurs noms Understat de la même équipe :", int(correspondance_joueurs.dropna(subset=['id_joueur']).duplicated(['equipe_us', 'id_joueur']).sum()))  # contrôle : on attend 0
a_verifier = correspondance_joueurs[correspondance_joueurs['methode'] != 'identique']  # tout ce qui n'est pas identique
a_verifier.sort_values(['methode', 'equipe_us'])  # affiche à valider

Noms encore avec un code web dans les tirs : 0 (attendu 0)
methode
identique                      494
nom de famille (à vérifier)     26
proche (à vérifier)             11
aucun                            3
Même joueur Sofascore associé à plusieurs noms Understat de la même équipe : 0


,equipe_us,joueur_us,id_joueur,joueur_sofa,methode
183,Juventus,Pierre Kalulu Kyatengwa,NaN,NaN,aucun
19,Napoli,Franck Zambo,NaN,NaN,aucun
35,Roma,Matìas Soulè Malvano,NaN,NaN,aucun
23,AC Milan,Estupiñán,805465.0,Pervis Estupiñán,nom de famille (à vérifier)
152,AC Milan,Cheveyo Mul,2205650.0,Cheveyo Mul-Balentien,nom de famille (à vérifier)
502,AC Milan,Alejandro Jiménez,1184308.0,Álex Jiménez,nom de famille (à vérifier)
50,Cagliari,Zito,1000150.0,Zito Luvumbo,nom de famille (à vérifier)
211,Cagliari,Ndary Adopo,928777.0,Michel Adopo,nom de famille (à vérifier)
442,Cagliari,Sulemana,105905.0,Ibrahim Sulemana,nom de famille (à vérifier)
58,Como,Tasos Douvikas,894863.0,Anastasios Douvikas,nom de famille (à vérifier)


In [19]:
manuel = {  # cas réglés à la main : (équipe Understat, nom Understat) vers un morceau du nom Sofascore, sans accent
    ('Juventus', 'Pierre Kalulu Kyatengwa'): 'kalulu',  # Pierre Kalulu
    ('Napoli', 'Franck Zambo'): 'anguissa',  # Frank Anguissa : Understat l'écrit Franck Zambo
    ('Roma', 'Matìas Soulè Malvano'): 'soule',  # Matías Soulé
}
for (eq, nom_us), morceau in manuel.items():  # parcourt les cas manuels
    eff = effectifs[effectifs['id_equipe'] == us_vers_so[eq]].copy()  # effectif Sofascore de l'équipe
    cand = eff[eff['joueur'].map(normaliser).str.contains(morceau)]  # joueurs dont le nom contient ce morceau
    print(eq, '|', nom_us, '→ candidats :', list(cand['joueur']))  # affiche les candidats pour contrôle
    if len(cand) == 1:  # un seul candidat : on accepte
        masque = (correspondance_joueurs['equipe_us'] == eq) & (correspondance_joueurs['joueur_us'] == nom_us)  # ligne à corriger
        correspondance_joueurs.loc[masque, ['id_joueur', 'joueur_sofa', 'methode']] = [cand.iloc[0]['id_joueur'], cand.iloc[0]['joueur'], 'manuel']  # remplit la correspondance

print(correspondance_joueurs['methode'].value_counts().to_string())  # nouveaux comptes : plus aucun "aucun" attendu
print("Joueurs sans numéro Sofascore :", int(correspondance_joueurs['id_joueur'].isna().sum()), "(attendu 0)")  # contrôle final
correspondance_joueurs.to_csv('data/processed/correspondance_joueurs.csv', index=False)  # sauvegarde la table finale

Juventus | Pierre Kalulu Kyatengwa → candidats : ['Pierre Kalulu']
Napoli | Franck Zambo → candidats : ['Frank Anguissa']
Roma | Matìas Soulè Malvano → candidats : ['Matías Soulé']
methode
identique                      494
nom de famille (à vérifier)     26
proche (à vérifier)             11
manuel                           3
Joueurs sans numéro Sofascore : 0 (attendu 0)


## 12. Préparer la comparaison avec la ligue (Sofascore)

C'est le cœur de la méthode : un chiffre seul ne veut rien dire. 1,59 xG par match est beaucoup ou peu selon ce que font les 19 autres équipes. Chaque indicateur d'une équipe sera donc comparé aux 20 équipes de la ligue.

Trois précautions de lecture :
1. **Vide veut dire zéro.** Pour les statistiques de comptage (grosses occasions, cartons, erreurs), Sofascore n'écrit rien quand l'événement n'a pas eu lieu. On le remplace par 0, sinon on ignorerait les matchs sans événement.
2. **Les vrais taux sont recalculés.** Un « pourcentage » Sofascore est parfois un nombre de réussites. On recalcule le taux nous mêmes avec le total des tentatives.
3. **Ce que l'équipe subit** est lu dans les statistiques de l'adversaire du même match. Une équipe qui concède peu de tirs n'est pas seulement une équipe qui défend bien : on mesure le résultat, pas la cause.

Pour le xG de l'équipe, on utilise celui de Sofascore. Celui d'Understat dépasse les buts réellement marqués d'environ 17 % sur toute la ligue, un écart qui existe pour toutes les équipes. On ne parle donc jamais de « sur performance » d'une équipe sans le comparer à cette norme de la ligue.

In [20]:
EQUIPE_ID = us_vers_so[EQUIPE]  # numéro Sofascore de l'équipe d'exemple, retrouvé grâce à la table de correspondance (2692 pour l'AC Milan)

comptages = ['bigChanceCreated', 'bigChanceMissed', 'bigChanceScored', 'accurateThroughBall', 'errorsLeadToShot', 'errorsLeadToGoal',  # statistiques de comptage où un vide veut dire zéro événement
             'yellowCards', 'redCards', 'offsides', 'hitWoodwork', 'punches', 'highClaims', 'diveSaves', 'penaltySaves', 'blockedScoringAttempt', 'fouledFinalThird']  # suite de la liste
base = camps.copy()  # copie du tableau des 760 camps
for c in comptages:  # parcourt les statistiques de comptage
    if c in base.columns:  # si la colonne existe
        base[c] = base[c].fillna(0)  # vide veut dire aucun événement, donc 0

autre_cote = {'home': 'away', 'away': 'home'}  # pour retrouver le camp adverse d'un même match
base['cote_adverse'] = base['cote'].map(autre_cote)  # camp adverse de chaque ligne
adverse = base.drop(columns=['id_equipe', 'cote_adverse']).add_prefix('adv_')  # statistiques de l'adversaire, préfixées adv_
adverse = adverse.rename(columns={'adv_id_match': 'id_match', 'adv_cote': 'cote_adverse'})  # clés de jointure
dup = base.merge(adverse, on=['id_match', 'cote_adverse'], how='left')  # chaque ligne reçoit les statistiques de son adversaire

dup['tirs'] = dup['totalShotsOnGoal']  # tirs totaux (ce nom Sofascore désigne bien tous les tirs)
dup['pct_passes_reussies'] = 100 * dup['accuratePasses'] / dup['passes']  # taux de passes réussies, calculé nous-mêmes
dup['pct_duels_aeriens'] = 100 * dup['aerialDuelsPercentage'] / dup['aerialDuelsPercentage_total']  # vrai taux de duels aériens gagnés
dup['pct_duels_sol'] = 100 * dup['groundDuelsPercentage'] / dup['groundDuelsPercentage_total']  # vrai taux de duels au sol gagnés
dup['pct_dribbles'] = 100 * dup['dribblesPercentage'] / dup['dribblesPercentage_total']  # vrai taux de dribbles réussis
dup['pct_tacles'] = 100 * dup['wonTacklePercent'] / dup['wonTacklePercent_total']  # vrai taux de tacles gagnés
dup['pct_cadres'] = 100 * dup['shotsOnGoal'] / dup['tirs']  # part des tirs cadrés

## 13. La comparaison : moyenne, médiane, rang et percentile

Pour chaque indicateur, le tableau donne :
1. la valeur de l'équipe, par match ;
2. la moyenne et la médiane des 20 équipes ;
3. le rang, où 1 est la valeur la plus haute de la ligue ;
4. le percentile, c'est à dire la part des 19 autres équipes qui sont en dessous. Un percentile de 90 veut dire que l'équipe dépasse 90 % des autres.

Attention : un rang haut n'est pas toujours bon. Être 1er aux erreurs menant à un tir est un point faible. Le sens de chaque indicateur est donc précisé à l'étape suivante.

**Contrôle attendu** : 30 lignes pour Milan, dont l'xG par match à 1,59 contre 1,26 en moyenne dans la ligue (4e valeur la plus haute).

In [21]:
indicateurs = {  # nom affiché : colonne du tableau, groupés par bloc du rapport
    'Style : possession (%)': 'ballPossession', 'Style : passes par match': 'passes', 'Style : passes réussies (%)': 'pct_passes_reussies',
    'Style : entrées dans le dernier tiers': 'finalThirdEntries', 'Style : longs ballons réussis': 'accurateLongBalls',
    'Attaque : xG': 'expectedGoals', 'Attaque : tirs': 'tirs', 'Attaque : tirs cadrés': 'shotsOnGoal', 'Attaque : tirs dans la surface': 'totalShotsInsideBox',
    'Attaque : ballons touchés dans la surface adverse': 'touchesInOppBox', 'Attaque : grosses occasions créées': 'bigChanceCreated',
    'Attaque : centres réussis': 'accurateCross', 'Attaque : dribbles réussis (%)': 'pct_dribbles',
    'Défense : tacles': 'totalTackle', 'Défense : interceptions': 'interceptionWon', 'Défense : dégagements': 'totalClearance',
    'Défense : récupérations': 'ballRecovery', 'Défense : duels aériens gagnés (%)': 'pct_duels_aeriens', 'Défense : duels au sol gagnés (%)': 'pct_duels_sol',
    'Défense : erreurs menant à un tir': 'errorsLeadToShot',
    'Discipline : fautes': 'fouls', 'Discipline : cartons jaunes': 'yellowCards', 'Discipline : cartons rouges': 'redCards',
}
subis = {  # ce que l'équipe concède, lu dans les statistiques de l'adversaire
    'Subi : xG concédé': 'adv_expectedGoals', 'Subi : tirs concédés': 'adv_totalShotsOnGoal', 'Subi : tirs cadrés concédés': 'adv_shotsOnGoal',
    'Subi : tirs concédés dans la surface': 'adv_totalShotsInsideBox', 'Subi : ballons touchés dans notre surface': 'adv_touchesInOppBox',
    'Subi : grosses occasions concédées': 'adv_bigChanceCreated', 'Subi : possession adverse (%)': 'adv_ballPossession',
}
tous_ind = {**indicateurs, **subis}  # fusionne les deux dictionnaires

moy_equipes = dup.groupby('id_equipe')[list(tous_ind.values())].mean()  # moyenne par match de chaque équipe sur chaque indicateur
moy_equipes.columns = list(tous_ind.keys())  # remplace les noms techniques par les noms affichés

def comparer_equipe(id_equipe, table=moy_equipes):  # fonction réutilisable pour n'importe quelle équipe
    lignes = []  # lignes du résultat
    for ind in table.columns:  # parcourt les indicateurs
        serie = table[ind]  # moyennes des 20 équipes sur cet indicateur
        valeur = serie[id_equipe]  # valeur de l'équipe choisie
        lignes.append({  # une ligne par indicateur
            'indicateur': ind,  # nom affiché
            'équipe': round(valeur, 2),  # valeur de l'équipe
            'moyenne ligue': round(serie.mean(), 2),  # moyenne des 20 équipes
            'médiane ligue': round(serie.median(), 2),  # médiane des 20 équipes
            'rang (1 = plus haut)': int(serie.rank(ascending=False, method='min')[id_equipe]),  # place de l'équipe, 1 = valeur la plus haute
            'percentile': int(round(100 * (serie < valeur).sum() / (len(serie) - 1))),  # part des 19 autres équipes strictement en dessous
        })
    return pd.DataFrame(lignes)  # tableau final

comparaison = comparer_equipe(EQUIPE_ID)  # lance la comparaison pour l'équipe d'exemple
pd.set_option('display.width', 200)  # élargit l'affichage
pd.set_option('display.max_rows', 100)  # permet d'afficher toutes les lignes
comparaison  # affiche le tableau

,indicateur,équipe,moyenne ligue,médiane ligue,rang (1 = plus haut),percentile
0,Style : possession (%),51.84,50.00,49.53,8,63
1,Style : passes par match,483.68,431.50,412.53,8,63
2,Style : passes réussies (%),86.76,81.87,82.28,3,89
3,Style : entrées dans le dernier tiers,50.45,51.47,51.16,13,37
4,Style : longs ballons réussis,22.53,22.56,22.72,12,42
5,Attaque : xG,1.59,1.26,1.18,4,84
6,Attaque : tirs,13.24,12.31,11.68,5,74
7,Attaque : tirs cadrés,4.37,4.03,3.83,7,68
8,Attaque : tirs dans la surface,9.16,7.92,7.22,6,74
9,Attaque : ballons touchés dans la surface adverse,26.37,21.91,20.07,5,79


## 14. Les conclusions : points forts, points faibles, style

On transforme le tableau en phrases, avec des règles écrites à l'avance pour ne pas choisir les chiffres qui arrangent :
1. Un indicateur n'est signalé que si l'équipe est dans les **5 premières ou les 5 dernières** de la ligue.
2. **Et** si l'écart à la moyenne est d'au moins une demi unité de dispersion de la ligue (écart type). Cela évite de signaler une différence minuscule dans une ligue où tout le monde se ressemble.
3. Chaque indicateur a un sens : « haut » (plus c'est haut, mieux c'est), « bas » (plus c'est haut, pire c'est) ou « style » (décrit une façon de jouer sans la juger, comme les passes réussies ou les fautes).

Une équipe n'a donc pas de score. Le nombre de points forts et de points faibles ne se compare pas d'une équipe à l'autre, car plusieurs indicateurs racontent la même chose (par exemple xG et tirs dans la surface).

**Contrôle attendu pour Milan** : parmi les points forts les grosses occasions créées et les centres réussis, et comme point faible les erreurs menant à un tir (0,76 par match contre 0,48 en moyenne, 1re valeur la plus haute sur 20).

In [22]:
sens = {  # sens de lecture de chaque indicateur : 'haut' = plus c'est haut mieux c'est, 'bas' = plus c'est haut pire c'est, 'style' = décrit sans juger
    'Style : possession (%)': 'style', 'Style : passes par match': 'style', 'Style : passes réussies (%)': 'style',
    'Style : entrées dans le dernier tiers': 'style', 'Style : longs ballons réussis': 'style',
    'Attaque : xG': 'haut', 'Attaque : tirs': 'haut', 'Attaque : tirs cadrés': 'haut', 'Attaque : tirs dans la surface': 'haut',
    'Attaque : ballons touchés dans la surface adverse': 'haut', 'Attaque : grosses occasions créées': 'haut',
    'Attaque : centres réussis': 'haut', 'Attaque : dribbles réussis (%)': 'haut',
    'Défense : tacles': 'style', 'Défense : interceptions': 'style', 'Défense : dégagements': 'style', 'Défense : récupérations': 'style',
    'Défense : duels aériens gagnés (%)': 'haut', 'Défense : duels au sol gagnés (%)': 'haut', 'Défense : erreurs menant à un tir': 'bas',
    'Discipline : fautes': 'style', 'Discipline : cartons jaunes': 'bas', 'Discipline : cartons rouges': 'bas',
    'Subi : xG concédé': 'bas', 'Subi : tirs concédés': 'bas', 'Subi : tirs cadrés concédés': 'bas', 'Subi : tirs concédés dans la surface': 'bas',
    'Subi : ballons touchés dans notre surface': 'bas', 'Subi : grosses occasions concédées': 'bas',
}
TOP_N = 5  # nombre de places en haut et en bas du classement qui comptent comme "se démarque"
SEUIL_ECART = 0.5  # écart minimal à la moyenne, en écarts-types de la ligue, pour éviter de signaler de petites différences

def virgule(x):  # écrit un nombre avec la virgule française
    return str(round(x, 2)).replace('.', ',')  # arrondit à 2 décimales puis remplace le point

def conclusions_ligue(id_equipe, table=moy_equipes):  # fonction réutilisable pour n'importe quelle équipe
    comp = comparer_equipe(id_equipe, table[[c for c in table.columns if c in sens]])  # tableau de comparaison, sans la possession adverse
    comp['écart-type ligue'] = comp['indicateur'].map(lambda i: table[i].std())  # dispersion des 20 équipes sur chaque indicateur
    comp['z'] = (comp['équipe'] - comp['moyenne ligue']) / comp['écart-type ligue']  # écart à la moyenne en nombre d'écarts-types
    lignes = []  # conclusions trouvées
    for _, r in comp.iterrows():  # parcourt les indicateurs
        rang = int(r['rang (1 = plus haut)'])  # place de l'équipe, 1 = valeur la plus haute
        haut = rang <= TOP_N  # dans les 5 premières valeurs de la ligue
        bas = rang >= 20 - TOP_N + 1  # dans les 5 dernières valeurs de la ligue
        if not (haut or bas) or abs(r['z']) < SEUIL_ECART:  # ni dans le haut ni dans le bas, ou écart trop faible
            continue  # on ne dit rien sur cet indicateur
        s = sens[r['indicateur']]  # sens de lecture de l'indicateur
        if s == 'style':  # indicateur qui décrit sans juger
            etiquette = 'Style'  # simple caractéristique
        elif (s == 'haut') == haut:  # haut et bon, ou bas et mauvais, donc favorable
            etiquette = 'Point fort'  # favorable à l'équipe
        else:  # défavorable
            etiquette = 'Point faible'  # défavorable à l'équipe
        unite = ' %' if '(%)' in r['indicateur'] else ' par match'  # unité : pourcentage ou par match
        rang_cote = rang if haut else 21 - rang  # place comptée depuis le côté concerné
        ordinal = '1re' if rang_cote == 1 else f"{rang_cote}e"  # écriture 1re, 2e, 3e
        sens_txt = 'haute' if haut else 'basse'  # valeur la plus haute ou la plus basse
        bloc, nom = r['indicateur'].split(' : ', 1)  # sépare le bloc du rapport et le nom de l'indicateur
        lignes.append({'type': etiquette, 'indicateur': r['indicateur'], 'texte': f"{bloc} : {nom}, {virgule(r['équipe'])}{unite} contre {virgule(r['moyenne ligue'])}{unite} en moyenne dans la ligue ({ordinal} valeur la plus {sens_txt} sur 20)."})  # phrase générée à partir des chiffres
    return pd.DataFrame(lignes)  # tableau des conclusions

conclusions_exemple = conclusions_ligue(EQUIPE_ID)  # lance pour l'équipe d'exemple
for t in ['Point fort', 'Point faible', 'Style']:  # affiche par catégorie
    print(f"--- {t} ---")  # titre de la catégorie
    for phrase in conclusions_exemple[conclusions_exemple['type'] == t]['texte']:  # parcourt les phrases
        print("•", phrase)  # affiche la phrase

--- Point fort ---
• Attaque : xG, 1,59 par match contre 1,26 par match en moyenne dans la ligue (4e valeur la plus haute sur 20).
• Attaque : ballons touchés dans la surface adverse, 26,37 par match contre 21,91 par match en moyenne dans la ligue (5e valeur la plus haute sur 20).
• Attaque : grosses occasions créées, 2,63 par match contre 2,07 par match en moyenne dans la ligue (3e valeur la plus haute sur 20).
• Attaque : centres réussis, 5,08 par match contre 4,26 par match en moyenne dans la ligue (2e valeur la plus haute sur 20).
• Défense : duels au sol gagnés (%), 52,21 % contre 50,0 % en moyenne dans la ligue (2e valeur la plus haute sur 20).
• Subi : grosses occasions concédées, 1,66 par match contre 2,07 par match en moyenne dans la ligue (4e valeur la plus basse sur 20).
--- Point faible ---
• Défense : erreurs menant à un tir, 0,76 par match contre 0,48 par match en moyenne dans la ligue (1re valeur la plus haute sur 20).
--- Style ---
• Style : passes réussies (%), 86,76 %

## 15. Coups de pied arrêtés et penaltys

Les coups de pied arrêtés sont une source de buts que l'on peut préparer. Pour chaque équipe, on sépare les tirs et les buts **marqués** et **encaissés** selon la situation de jeu : corner, coup franc direct, autre coup de pied arrêté, penalty et jeu ouvert. Puis on compare l'équipe aux 19 autres, exactement comme pour les autres indicateurs.

Deux précautions :
1. Un but contre son camp est attribué à l'équipe qui en profite (colonne `equipe_but`), et n'est pas compté comme un tir.
2. Les nombres sont petits (quelques penaltys par équipe sur la saison). Une différence de un ou deux événements ne prouve rien, et le rapport doit le dire.

**Contrôles attendus** : 922 buts dans les tirs et 922 dans les scores Sofascore (écart de 0), et 922 buts classés dans les tableaux par situation.

In [23]:
buts_tirs = int(tirs_ligue['result'].isin(['Goal', 'OwnGoal']).sum())  # buts comptés dans le fichier de tirs, y compris les buts contre son camp
buts_scores = int(matchs_ligue['buts_dom'].sum() + matchs_ligue['buts_ext'].sum())  # buts comptés dans les scores Sofascore
print("Buts dans les tirs :", buts_tirs, "| buts dans les scores :", buts_scores, "| écart :", buts_tirs - buts_scores)  # contrôle : on attend un écart de 0

t = tirs_ligue.copy()  # copie du fichier de tirs
t['est_but'] = t['result'].isin(['Goal', 'OwnGoal'])  # True si le tir a fini en but (buts contre son camp inclus)
t['equipe_encaisse'] = np.where(t['equipe_but'] == t['equipe'], t['adversaire'], t['equipe'])  # l'équipe qui encaisse le but, à l'opposé de celle créditée
t['est_tir'] = t['result'] != 'OwnGoal'  # un but contre son camp n'est pas un tir de l'équipe

types = {'FromCorner': 'corner', 'DirectFreekick': 'coup franc direct', 'SetPiece': 'autre coup de pied arrêté', 'Penalty': 'penalty', 'OpenPlay': 'jeu ouvert'}  # noms lisibles des situations
lignes = []  # indicateurs par équipe
for equipe in sorted(t['equipe'].unique()):  # parcourt les 20 équipes
    ligne = {'equipe': equipe}  # une ligne par équipe
    for sit, nom in types.items():  # parcourt chaque type de situation
        pour = t[(t['situation'] == sit) & (t['equipe'] == equipe) & t['est_tir']]  # tirs de l'équipe
        contre = t[(t['situation'] == sit) & (t['adversaire'] == equipe) & t['est_tir']]  # tirs subis
        ligne[f'{nom} : tirs'] = len(pour)  # tirs tentés
        ligne[f'{nom} : xG'] = round(pour['xG'].sum(), 2)  # qualité des tirs
        ligne[f'{nom} : buts'] = int(t[(t['situation'] == sit) & (t['equipe_but'] == equipe) & t['est_but']].shape[0])  # buts marqués, buts contre son camp adverses inclus
        ligne[f'{nom} subi : tirs'] = len(contre)  # tirs concédés
        ligne[f'{nom} subi : xG'] = round(contre['xG'].sum(), 2)  # qualité des tirs concédés
        ligne[f'{nom} subi : buts'] = int(t[(t['situation'] == sit) & (t['equipe_encaisse'] == equipe) & t['est_but']].shape[0])  # buts encaissés
    lignes.append(ligne)  # ajoute la ligne
cpa_ligue = pd.DataFrame(lignes).set_index('equipe')  # tableau final, une ligne par équipe

cpa_noms = ['corner', 'coup franc direct', 'autre coup de pied arrêté']  # coups de pied arrêtés hors penalty
for suffixe in ['', ' subi']:  # pour et contre
    for stat in ['tirs', 'xG', 'buts']:  # trois mesures
        cpa_ligue[f'coups de pied arrêtés hors penalty{suffixe} : {stat}'] = sum(cpa_ligue[f'{n}{suffixe} : {stat}'] for n in cpa_noms)  # somme des trois types
cpa_ligue['part des buts marqués sur coup de pied arrêté (%)'] = 100 * (cpa_ligue['coups de pied arrêtés hors penalty : buts'] + cpa_ligue['penalty : buts']) / (cpa_ligue[[f'{n} : buts' for n in types.values()]].sum(axis=1))  # part des buts venus d'un corner, coup franc ou penalty
cpa_ligue['part des buts encaissés sur coup de pied arrêté (%)'] = 100 * (cpa_ligue['coups de pied arrêtés hors penalty subi : buts'] + cpa_ligue['penalty subi : buts']) / (cpa_ligue[[f'{n} subi : buts' for n in types.values()]].sum(axis=1))  # même calcul pour les buts encaissés

cpa_ligue.to_csv('data/processed/cpa_ligue.csv')  # sauvegarde
colonnes_cpa = ['corner : tirs', 'corner : buts', 'corner subi : tirs', 'corner subi : buts', 'coup franc direct : tirs', 'coup franc direct : buts',  # colonnes principales à afficher
                'coup franc direct subi : buts', 'penalty : tirs', 'penalty : buts', 'penalty subi : tirs', 'penalty subi : buts',
                'part des buts marqués sur coup de pied arrêté (%)', 'part des buts encaissés sur coup de pied arrêté (%)']
print("Total buts classés dans les tableaux :", int(cpa_ligue[[f'{n} : buts' for n in types.values()]].sum().sum()), "(attendu", buts_scores, ")")  # contrôle : tous les buts sont classés
cpa_comp = comparer_equipe(EQUIPE, cpa_ligue[colonnes_cpa])  # même comparaison que pour les stats Sofascore (valeurs sur la saison)
cpa_comp.round(1)  # affiche le tableau

Buts dans les tirs : 922 | buts dans les scores : 922 | écart : 0
Total buts classés dans les tableaux : 922 (attendu 922 )


,indicateur,équipe,moyenne ligue,médiane ligue,rang (1 = plus haut),percentile
0,corner : tirs,77.0,73.2,69.0,8,63
1,corner : buts,6.0,6.5,6.5,11,42
2,corner subi : tirs,62.0,73.2,78.5,14,32
3,corner subi : buts,6.0,6.5,6.0,10,26
4,coup franc direct : tirs,9.0,14.0,13.0,16,16
5,coup franc direct : buts,0.0,0.6,0.5,11,0
6,coup franc direct subi : buts,0.0,0.6,0.0,9,0
7,penalty : tirs,9.0,5.3,5.0,1,95
8,penalty : buts,7.0,4.0,4.5,1,100
9,penalty subi : tirs,8.0,5.3,6.0,2,95


## 16. Joueurs offensifs les plus dangereux

À partir des tirs, on mesure l'implication offensive de chaque joueur, pour la saison :
1. **Ce qu'il tire** : nombre de tirs, xG (qualité des tirs), buts.
2. **Ce qu'il crée** : passes menant à un tir, xG des tirs créés, passes décisives.
3. **La menace totale** : xG de ses tirs hors penalty plus xG des tirs qu'il a créés. Les penaltys sont écartés, car ils ne disent rien du jeu et dépendent surtout de qui les tire.

On mesure aussi la **dépendance** : si un joueur a marqué au moins 25 % des buts hors penalty de l'équipe, le rapport le signale. Ce seuil est un choix de lecture et reste modifiable. Il veut simplement dire « un buteur sur quatre ».

Attention : on ne compare pas ici les buts au xG pour dire qu'un joueur est « en réussite » ou « en échec ». L'xG d'Understat surestime les buts sur toute la ligue, donc l'écart ne signifierait rien sans la norme de la ligue.

In [24]:
def joueurs_dangereux(equipe, tirs=tirs_ligue):  # fonction réutilisable : les joueurs offensifs d'une équipe, à partir des tirs
    d = tirs[(tirs['equipe'] == equipe) & (tirs['result'] != 'OwnGoal')].copy()  # tirs de l'équipe, sans les buts contre son camp
    d['but'] = d['result'] == 'Goal'  # True si le tir est un but
    d['hors_penalty'] = d['situation'] != 'Penalty'  # True si ce n'est pas un penalty
    buts_equipe = int(((tirs['equipe_but'] == equipe) & tirs['result'].isin(['Goal', 'OwnGoal'])).sum())  # buts de l'équipe, buts adverses contre son camp inclus
    buts_equipe_hp = int(((tirs['equipe_but'] == equipe) & (tirs['result'].isin(['Goal', 'OwnGoal'])) & (tirs['situation'] != 'Penalty')).sum())  # mêmes buts sans les penaltys

    tireurs = d.groupby('player').agg(  # une ligne par joueur qui a tiré
        tirs=('result', 'size'),  # nombre de tirs
        xG=('xG', 'sum'),  # qualité cumulée des tirs
        buts=('but', 'sum'),  # buts marqués
    )
    hp = d[d['hors_penalty']].groupby('player').agg(tirs_hp=('result', 'size'), xG_hp=('xG', 'sum'), buts_hp=('but', 'sum'))  # mêmes mesures sans les penaltys
    pen = d[~d['hors_penalty']].groupby('player').size().rename('penaltys_tires')  # penaltys tirés par joueur
    crea = d[d['player_assisted'].notna() & (d['player_assisted'] != '')].groupby('player_assisted').agg(  # tirs de l'équipe provoqués par une passe
        passes_vers_tir=('result', 'size'),  # nombre de passes menant à un tir
        xG_crees=('xG', 'sum'),  # qualité des tirs ainsi créés
        passes_decisives=('but', 'sum'),  # passes menant à un but
    )
    crea.index.name = 'player'  # même nom d'index que les tireurs pour la jointure
    tab = tireurs.join(hp, how='outer').join(pen, how='outer').join(crea, how='outer').fillna(0)  # regroupe tireurs, créateurs et tireurs de penalty
    tab['part_buts_equipe_%'] = 100 * tab['buts'] / buts_equipe if buts_equipe else 0  # part de tous les buts de l'équipe
    tab['part_buts_hors_penalty_%'] = 100 * tab['buts_hp'] / buts_equipe_hp if buts_equipe_hp else 0  # part des buts de l'équipe hors penalty
    tab['xG_total_menace'] = tab['xG_hp'] + tab['xG_crees']  # menace totale : tirs hors penalty plus tirs créés pour les autres
    return tab.round(2).sort_values('xG_total_menace', ascending=False)  # trie du plus dangereux au moins dangereux

dangereux = joueurs_dangereux(EQUIPE)  # lance pour l'équipe d'exemple
colonnes = ['tirs', 'xG', 'buts', 'buts_hp', 'part_buts_hors_penalty_%', 'penaltys_tires', 'passes_vers_tir', 'xG_crees', 'passes_decisives', 'xG_total_menace']  # colonnes à afficher
print("Buts de", EQUIPE, ":", int(dangereux['buts'].sum()), "| penaltys tirés :", int(dangereux['penaltys_tires'].sum()))  # contrôle : buts des joueurs, sans les buts contre son camp
dangereux[colonnes].head(10)  # les 10 joueurs les plus dangereux

Buts de AC Milan : 53 | penaltys tirés : 9


,tirs,xG,buts,buts_hp,part_buts_hors_penalty_%,penaltys_tires,passes_vers_tir,xG_crees,passes_decisives,xG_total_menace
player,,,,,,,,,,
Christian Pulisic,54.0,9.54,8.0,8.0,17.39,1.0,37.0,5.04,4.0,13.82
Rafael Leão,63.0,9.60,9.0,7.0,15.22,2.0,23.0,4.98,3.0,13.06
Alexis Saelemaekers,43.0,3.60,3.0,3.0,6.52,0.0,53.0,4.79,3.0,8.39
Adrien Rabiot,46.0,4.36,6.0,6.0,13.04,0.0,27.0,3.94,4.0,8.30
Youssouf Fofana,44.0,4.59,2.0,2.0,4.35,0.0,36.0,3.64,3.0,8.22
Christopher Nkunku,29.0,9.94,7.0,2.0,4.35,5.0,16.0,1.61,3.0,7.75
Davide Bartesaghi,22.0,1.37,2.0,2.0,4.35,0.0,44.0,3.91,0.0,5.29
Luka Modric,20.0,1.25,2.0,2.0,4.35,0.0,50.0,4.03,3.0,5.27
Ruben Loftus-Cheek,29.0,4.14,3.0,3.0,6.52,0.0,10.0,1.04,1.0,5.18


In [25]:
SEUIL_DEPENDANCE = 25  # part des buts hors penalty, en pourcentage, à partir de laquelle on parle de dépendance à un joueur


def conclusions_joueurs(equipe, tab):  # génère les phrases du bloc joueurs à partir du tableau des joueurs dangereux
    phrases = []  # liste des phrases trouvées
    top = tab.sort_values('xG_total_menace', ascending=False).head(3)  # les 3 joueurs les plus impliqués dans les tirs
    phrases.append("Joueurs les plus dangereux (tirs hors penalty et tirs provoqués) : " + ", ".join(f"{n} ({r['xG_total_menace']:.1f})" for n, r in top.iterrows()) + ".")  # phrase de synthèse

    premier = tab.sort_values('part_buts_hors_penalty_%', ascending=False).iloc[0]  # le joueur qui marque la plus grande part des buts hors penalty
    if premier['part_buts_hors_penalty_%'] >= SEUIL_DEPENDANCE:  # seuil de dépendance à un joueur
        phrases.append(f"Dépendance : {premier.name} a marqué {premier['part_buts_hors_penalty_%']:.0f} % des buts hors penalty de l'équipe.")  # alerte dépendance
    else:  # aucun joueur au-dessus du seuil
        phrases.append(f"Pas de dépendance à un joueur : le meilleur buteur ({premier.name}) fournit {premier['part_buts_hors_penalty_%']:.0f} % des buts hors penalty, sous le seuil de {SEUIL_DEPENDANCE} %.")  # buts répartis

    total_pen = int(tab['penaltys_tires'].sum())  # penaltys tirés par l'équipe
    if total_pen >= 3:  # assez de penaltys pour parler de tireur habituel
        tireur = tab.sort_values('penaltys_tires', ascending=False).iloc[0]  # joueur qui en a tiré le plus
        if tireur['penaltys_tires'] >= total_pen / 2:  # au moins la moitié des penaltys
            phrases.append(f"Tireur de penalty habituel : {tireur.name} ({int(tireur['penaltys_tires'])} penaltys sur {total_pen}).")  # tireur identifié
    createurs = tab.sort_values('passes_vers_tir', ascending=False).head(2)  # les 2 joueurs qui provoquent le plus de tirs
    phrases.append("Principaux créateurs de tirs : " + ", ".join(f"{n} ({int(r['passes_vers_tir'])} passes menant à un tir)" for n, r in createurs.iterrows()) + ".")  # phrase de synthèse des créateurs
    return phrases  # liste des phrases

for p in conclusions_joueurs(EQUIPE, dangereux):  # lance pour l'équipe d'exemple
    print("•", p)  # affiche chaque phrase

• Joueurs les plus dangereux (tirs hors penalty et tirs provoqués) : Christian Pulisic (13.8), Rafael Leão (13.1), Alexis Saelemaekers (8.4).
• Pas de dépendance à un joueur : le meilleur buteur (Christian Pulisic) fournit 17 % des buts hors penalty, sous le seuil de 25 %.
• Tireur de penalty habituel : Christopher Nkunku (5 penaltys sur 9).
• Principaux créateurs de tirs : Alexis Saelemaekers (53 passes menant à un tir), Luka Modric (50 passes menant à un tir).


## 17. L'effectif : minutes, rôle et menace

Pour savoir qui joue vraiment, on part des minutes (Sofascore). Chaque joueur reçoit un rôle selon sa part de minutes sur la saison :
1. **indispensable** : au moins 60 % des minutes ;
2. **peu utilisé** : moins de 30 % des minutes ;
3. entre les deux, **rotation** si le joueur fait au moins 45 minutes par match joué, sinon **remplaçant important**.

Ces seuils sont des choix de lecture. On y ajoute la menace offensive de la partie précédente, rattachée à chaque joueur grâce à la table de correspondance. La **menace par 90 minutes** n'est calculée que pour les joueurs ayant au moins 270 minutes (3 matchs complets) : en dessous, un taux est trop instable pour être lu.

**Contrôle attendu pour Milan** : 26 joueurs et 37 642 minutes au total, pour un maximum théorique de 37 620 (11 joueurs fois 3 420 minutes), l'écart venant des prolongations et des arrêts de jeu.

In [26]:
MIN_MINUTES_TAUX = 270  # minutes minimales (3 matchs complets) pour calculer un taux par 90 minutes
MINUTES_SAISON = 38 * 90  # minutes disponibles sur une saison de 38 matchs
SEUIL_INDISPENSABLE = 0.60  # part des minutes à partir de laquelle un joueur est indispensable (validé)
SEUIL_PEU_UTILISE = 0.30  # sous cette part des minutes, joueur peu utilisé (validé)
SEUIL_MIN_PAR_MATCH = 45  # minutes par match joué qui sépare rotation et remplaçant important (validé)

def fiche_joueurs(equipe_us, tirs=tirs_ligue):  # fonction réutilisable : minutes, rôle et menace de chaque joueur d'une équipe
    id_eq = us_vers_so[equipe_us]  # numéro Sofascore de l'équipe
    j = joueurs_ligue[joueurs_ligue['id_equipe'] == id_eq]  # lignes joueur-match de l'équipe
    minutes = j.groupby('id_joueur').agg(  # une ligne par joueur de l'effectif
        joueur=('joueur', 'first'),  # nom Sofascore
        poste=('poste', lambda s: s.mode().iloc[0]),  # poste le plus fréquent
        minutes=('minutesPlayed', 'sum'),  # minutes jouées
        matchs=('id_match', 'nunique'),  # matchs joués
        titularisations=('remplacant', lambda s: int((~s.astype(bool)).sum())),  # matchs commencés
    )
    minutes['part_minutes'] = (minutes['minutes'] / MINUTES_SAISON).round(2)  # part des minutes de la saison
    minutes['min_par_match'] = (minutes['minutes'] / minutes['matchs']).round(0)  # minutes par match joué

    def role(r):  # classe chaque joueur selon les seuils validés
        if r['part_minutes'] >= SEUIL_INDISPENSABLE:  # beaucoup de minutes
            return 'indispensable'  # titulaire incontournable
        if r['part_minutes'] < SEUIL_PEU_UTILISE:  # peu de minutes
            return 'peu utilisé'  # joue peu
        return 'rotation' if r['min_par_match'] >= SEUIL_MIN_PAR_MATCH else 'remplaçant important'  # entre les deux : selon la durée par match
    minutes['role'] = minutes.apply(role, axis=1)  # applique la règle

    corr = correspondance_joueurs[(correspondance_joueurs['equipe_us'] == equipe_us)].dropna(subset=['id_joueur']).copy()  # correspondance de cette équipe
    corr['id_joueur'] = corr['id_joueur'].astype(int)  # numéros en entiers
    danger = joueurs_dangereux(equipe_us, tirs).reset_index().rename(columns={'player': 'joueur_us'})  # tableau de menace, noms Understat
    danger = danger.merge(corr[['joueur_us', 'id_joueur']], on='joueur_us', how='left')  # ajoute le numéro Sofascore
    danger = danger.dropna(subset=['id_joueur']).copy()  # retire les joueurs non rattachés (aucun attendu)
    danger['id_joueur'] = danger['id_joueur'].astype(int)  # numéros en entiers
    danger = danger.groupby('id_joueur')[['tirs_hp', 'xG_hp', 'buts_hp', 'penaltys_tires', 'passes_vers_tir', 'xG_crees', 'xG_total_menace']].sum()  # un seul bloc par joueur

    fiche = minutes.join(danger, how='left').fillna({'tirs_hp': 0, 'xG_hp': 0, 'buts_hp': 0, 'penaltys_tires': 0, 'passes_vers_tir': 0, 'xG_crees': 0, 'xG_total_menace': 0})  # joint minutes et menace
    fiche['menace_90'] = np.where(fiche['minutes'] >= MIN_MINUTES_TAUX, fiche['xG_total_menace'] / fiche['minutes'] * 90, np.nan).round(2)  # menace par 90 minutes, seulement au-dessus du seuil
    return fiche.sort_values('minutes', ascending=False)  # trie par minutes jouées

fiche = fiche_joueurs(EQUIPE)  # lance pour l'équipe d'exemple
print("Joueurs :", len(fiche), "| somme des minutes :", int(fiche['minutes'].sum()), "(environ 11 x 3420 = 37620 attendu)")  # contrôle : les minutes de l'équipe sur la saison
print(fiche['role'].value_counts().to_string())  # nombre de joueurs par rôle
fiche[['joueur', 'poste', 'minutes', 'matchs', 'titularisations', 'part_minutes', 'role', 'tirs_hp', 'xG_total_menace', 'menace_90']].head(25)  # les 25 joueurs les plus utilisés

Joueurs : 26 | somme des minutes : 37642 (environ 11 x 3420 = 37620 attendu)
role
peu utilisé             10
indispensable            9
rotation                 4
remplaçant important     3


,joueur,poste,minutes,matchs,titularisations,part_minutes,role,tirs_hp,xG_total_menace,menace_90
id_joueur,,,,,,,,,,
191210,Mike Maignan,G,3296.0,37,37,0.96,indispensable,0.0,0.00,0.00
965893,Strahinja Pavlović,D,2870.0,34,33,0.84,indispensable,26.0,4.63,0.15
15466,Luka Modrić,M,2821.0,34,32,0.82,indispensable,20.0,5.27,0.17
901891,Alexis Saelemaekers,M,2797.0,35,33,0.82,indispensable,43.0,8.39,0.27
788255,Fikayo Tomori,D,2563.0,33,31,0.75,indispensable,9.0,1.60,0.06
250737,Adrien Rabiot,M,2550.0,29,28,0.75,indispensable,46.0,8.30,0.29
826195,Matteo Gabbia,D,2531.0,30,28,0.74,indispensable,19.0,1.60,0.06
1390489,Davide Bartesaghi,M,2442.0,31,27,0.71,indispensable,22.0,5.29,0.19
945062,Youssouf Fofana,M,2217.0,33,28,0.65,indispensable,44.0,8.22,0.33


## 18. Joueurs à surveiller

Deux listes, construites uniquement avec la menace par 90 minutes :
1. **Menace principale** : les trois meilleurs taux de l'équipe parmi les joueurs qui ont beaucoup joué (au moins 900 minutes, soit environ 10 matchs complets).
2. **Dangereux à l'entrée ou peu utilisés** : les joueurs qui figurent parmi les cinq meilleurs taux de l'équipe mais jouent peu.

Un joueur qui a au moins 900 minutes est placé dans le classement de la ligue, parmi les joueurs du même poste (attaquants, milieux, défenseurs). Un joueur qui a moins de minutes n'est pas classé : la phrase le précise, car un taux sur peu de matchs peut changer beaucoup avec un seul but.

In [27]:
MIN_MINUTES_LIGUE = 900  # minutes minimales pour entrer dans le classement de la ligue (environ 10 matchs complets)
POSTES = {'F': 'attaquants', 'M': 'milieux', 'D': 'défenseurs'}  # noms des postes pour les phrases

fiches_ligue = pd.concat({eq: fiche_joueurs(eq) for eq in correspondance_equipes['understat']}, names=['equipe_us', 'id_joueur']).reset_index()  # fiche des 20 équipes dans un seul tableau
eligibles = fiches_ligue[(fiches_ligue['minutes'] >= MIN_MINUTES_LIGUE) & (fiches_ligue['poste'] != 'G')].copy()  # joueurs de champ avec assez de minutes
print("Équipes :", fiches_ligue['equipe_us'].nunique(), "| joueurs de la ligue :", len(fiches_ligue), "| éligibles au classement :", len(eligibles))  # contrôle : 20 équipes

def position_ligue(valeur, poste):  # place d'un joueur parmi ceux de son poste dans la ligue
    groupe = eligibles[eligibles['poste'] == poste]['menace_90']  # menace par 90 minutes des joueurs du même poste
    return int((groupe > valeur).sum()) + 1, len(groupe)  # rang (1 = plus haute menace) et taille du groupe

def a_surveiller(equipe_us):  # fonction réutilisable : phrases de la liste à surveiller
    f = fiches_ligue[(fiches_ligue['equipe_us'] == equipe_us) & (fiches_ligue['poste'] != 'G') & fiches_ligue['menace_90'].notna()].copy()  # joueurs de champ de l'équipe avec un taux calculé
    phrases = []  # phrases à produire

    def texte(r):  # phrase commune à tous les joueurs
        taux = str(r['menace_90']).replace('.', ',')  # taux avec la virgule française
        if r['minutes'] >= MIN_MINUTES_LIGUE:  # assez de minutes pour situer le joueur dans la ligue
            rang, n = position_ligue(r['menace_90'], r['poste'])  # position dans la ligue
            nom_poste = POSTES.get(r['poste'], 'joueurs')  # nom du poste au pluriel
            situation = f"{rang}e sur {n} {nom_poste} de la ligue (au moins {MIN_MINUTES_LIGUE} minutes)"  # position dans le classement
        else:  # trop peu de minutes pour un classement
            situation = f"moins de {MIN_MINUTES_LIGUE} minutes jouées, classement non établi, taux à prendre avec prudence"  # avertissement neutre
        return f"{r['joueur']} ({r['poste']}) : {taux} de menace par 90 minutes, {situation}, {int(r['minutes'])} minutes jouées, rôle : {r['role']}."  # phrase générée à partir des chiffres

    titulaires = f[f['minutes'] >= MIN_MINUTES_LIGUE].sort_values('menace_90', ascending=False).head(3)  # les 3 meilleurs taux parmi ceux qui ont beaucoup joué
    phrases.append(('Menace principale', [texte(r) for _, r in titulaires.iterrows()]))  # bloc 1
    top5 = f.sort_values('menace_90', ascending=False).head(5).index  # les 5 meilleurs taux de l'équipe, tous temps de jeu confondus
    jokers = f[f.index.isin(top5) & ~f.index.isin(titulaires.index) & f['role'].isin(['remplaçant important', 'peu utilisé'])].sort_values('menace_90', ascending=False)  # bons taux mais peu de minutes, sans répéter le bloc 1
    phrases.append(('Dangereux à l\'entrée ou peu utilisés', [texte(r) for _, r in jokers.iterrows()]))  # bloc 2
    return phrases  # liste de blocs

for titre, lignes in a_surveiller(EQUIPE):  # lance pour l'équipe d'exemple
    print(f"--- {titre} ---")  # titre du bloc
    for l in lignes:  # parcourt les phrases
        print("•", l)  # affiche la phrase

Équipes : 20 | joueurs de la ligue : 609 | éligibles au classement : 308
--- Menace principale ---
• Christian Pulišić (F) : 0,77 de menace par 90 minutes, 5e sur 74 attaquants de la ligue (au moins 900 minutes), 1611 minutes jouées, rôle : rotation.
• Rafael Leão (F) : 0,63 de menace par 90 minutes, 11e sur 74 attaquants de la ligue (au moins 900 minutes), 1859 minutes jouées, rôle : rotation.
• Christopher Nkunku (F) : 0,51 de menace par 90 minutes, 21e sur 74 attaquants de la ligue (au moins 900 minutes), 1377 minutes jouées, rôle : remplaçant important.
--- Dangereux à l'entrée ou peu utilisés ---
• Niclas Füllkrug (F) : 0,69 de menace par 90 minutes, moins de 900 minutes jouées, classement non établi, taux à prendre avec prudence, 647 minutes jouées, rôle : peu utilisé.
• Santiago Giménez (F) : 0,44 de menace par 90 minutes, moins de 900 minutes jouées, classement non établi, taux à prendre avec prudence, 896 minutes jouées, rôle : peu utilisé.


## 19. Défense individuelle : points d'appui à tester

Même idée que pour les équipes : on compare chaque joueur de champ aux autres joueurs **du même poste** dans la ligue. On regarde six indicateurs :
1. dribbles subis par 90 minutes ;
2. ballons perdus par 90 minutes ;
3. fautes par 90 minutes ;
4. erreurs menant à un tir par 90 minutes ;
5. pourcentage de duels aériens gagnés ;
6. pourcentage de duels gagnés.

Pour chaque indicateur, on calcule le **percentile** du joueur dans son poste, orienté pour que 1 soit « le pire ». Un joueur est signalé s'il fait partie des 10 % les plus défavorables de son poste.

Quatre garde fous pour ne pas conclure sur du bruit :
1. au moins 900 minutes jouées pour entrer dans la comparaison ;
2. au moins 20 duels pour calculer un pourcentage de duels ;
3. au moins 3 erreurs pour signaler un joueur sur les erreurs, et le nombre brut d'erreurs est toujours écrit dans la phrase ;
4. le résultat est un **point d'appui à tester**, pas une certitude : les chiffres ne disent rien du placement réel du joueur.

In [28]:
colonnes_comptage = [c for c in ['aerialWon', 'aerialLost', 'duelWon', 'duelLost', 'challengeLost', 'dispossessed', 'fouls', 'errorLeadToAShot', 'errorLeadToAGoal'] if c in joueurs_ligue.columns]  # statistiques disponibles dans nos données
print("Statistiques utilisées :", colonnes_comptage)  # contrôle : on vérifie ce qui existe vraiment

jd = joueurs_ligue.copy()  # copie des lignes joueur-match
jd[colonnes_comptage] = jd[colonnes_comptage].fillna(0)  # un vide veut dire zéro événement
agg = jd.groupby(['id_equipe', 'id_joueur']).agg(  # un bloc par joueur et par équipe
    joueur=('joueur', 'first'), poste=('poste', lambda s: s.mode().iloc[0]), minutes=('minutesPlayed', 'sum'),  # identité et minutes
    **{c: (c, 'sum') for c in colonnes_comptage}  # somme de chaque statistique
).reset_index()
agg = agg[(agg['minutes'] >= MIN_MINUTES_LIGUE) & (agg['poste'].isin(['D', 'M', 'F']))].copy()  # joueurs de champ avec assez de minutes

agg['% duels aériens gagnés'] = np.where(agg['aerialWon'] + agg['aerialLost'] >= 20, 100 * agg['aerialWon'] / (agg['aerialWon'] + agg['aerialLost']), np.nan)  # taux aérien, seulement si assez de duels
agg['% duels gagnés'] = np.where(agg['duelWon'] + agg['duelLost'] >= 20, 100 * agg['duelWon'] / (agg['duelWon'] + agg['duelLost']), np.nan)  # taux de duels, seulement si assez de duels
for c, nom in [('challengeLost', 'dribbles subis par 90'), ('dispossessed', 'ballons perdus par 90'), ('fouls', 'fautes par 90'), ('errorLeadToAShot', 'erreurs menant à un tir par 90')]:  # taux par 90 minutes
    if c in agg.columns:  # si la statistique existe
        agg[nom] = agg[c] / agg['minutes'] * 90  # taux par 90 minutes

# pour chaque indicateur, True signifie que plus c'est haut, plus c'est mauvais
mauvais_si_haut = {'dribbles subis par 90': True, 'ballons perdus par 90': True, 'fautes par 90': True, 'erreurs menant à un tir par 90': True, '% duels aériens gagnés': False, '% duels gagnés': False}  # sens de lecture
for ind, haut_mauvais in mauvais_si_haut.items():  # parcourt les indicateurs
    if ind in agg.columns:  # si calculé
        pct = agg.groupby('poste')[ind].rank(pct=True, method='max')  # place dans le poste, entre 0 et 1
        agg['mauvais ' + ind] = (pct if haut_mauvais else 1 - pct + 1 / agg.groupby('poste')[ind].transform('count')).round(2)  # 1 = le pire du poste
print("Joueurs classés :", len(agg), "| par poste :", agg['poste'].value_counts().to_dict())  # contrôle

id_eq = us_vers_so[EQUIPE]  # numéro Sofascore de l'équipe analysée
vue = agg[(agg['id_equipe'] == id_eq)].copy()  # joueurs de l'équipe
cols = ['joueur', 'poste', 'minutes'] + [i for i in mauvais_si_haut if i in vue.columns] + ['mauvais ' + i for i in mauvais_si_haut if i in vue.columns]  # colonnes à afficher
vue.sort_values('minutes', ascending=False)[cols].round(1)  # tableau de l'équipe

Statistiques utilisées : ['aerialWon', 'aerialLost', 'duelWon', 'duelLost', 'challengeLost', 'dispossessed', 'fouls', 'errorLeadToAShot', 'errorLeadToAGoal']
Joueurs classés : 308 | par poste : {'M': 132, 'D': 102, 'F': 74}


,joueur,poste,minutes,dribbles subis par 90,ballons perdus par 90,fautes par 90,erreurs menant à un tir par 90,% duels aériens gagnés,% duels gagnés,mauvais dribbles subis par 90,mauvais ballons perdus par 90,mauvais fautes par 90,mauvais erreurs menant à un tir par 90,mauvais % duels aériens gagnés,mauvais % duels gagnés
163,Strahinja Pavlović,D,2870.0,0.5,0.3,0.9,0.0,55.6,57.3,0.6,0.8,0.3,0.4,0.5,0.4
146,Luka Modrić,M,2821.0,0.9,0.4,0.7,0.1,50.0,55.4,0.8,0.2,0.1,0.9,0.4,0.2
160,Alexis Saelemaekers,M,2797.0,1.0,0.6,0.8,0.1,45.5,58.2,0.8,0.4,0.2,0.9,0.5,0.1
153,Fikayo Tomori,D,2563.0,0.6,0.2,1.3,0.0,47.1,53.8,0.8,0.4,0.6,0.3,0.8,0.6
150,Adrien Rabiot,M,2550.0,0.7,1.2,1.2,0.0,61.1,56.2,0.4,0.8,0.5,0.6,0.1,0.1
157,Matteo Gabbia,D,2531.0,0.2,0.0,0.9,0.0,65.3,58.8,0.1,0.0,0.2,0.3,0.1,0.4
168,Davide Bartesaghi,M,2442.0,0.7,0.8,0.7,0.1,45.5,47.8,0.4,0.6,0.1,0.8,0.5,0.6
162,Youssouf Fofana,M,2217.0,0.8,1.0,0.9,0.2,31.2,41.7,0.6,0.7,0.2,1.0,0.9,0.9
158,Rafael Leão,F,1859.0,0.3,1.2,0.9,0.0,50.9,47.0,0.4,0.3,0.2,0.7,0.2,0.3
165,Koni De Winter,D,1805.0,0.2,0.1,1.3,0.1,65.6,59.4,0.1,0.3,0.7,0.8,0.1,0.3


In [29]:
SEUIL_MAUVAIS = 0.9  # percentile à partir duquel on signale un joueur (les 10 % pires de son poste)
MIN_ERREURS = 3  # nombre minimal d'erreurs pour signaler un joueur sur ce critère
libelles = {  # indicateur : (texte, côté signalé, nombre de décimales)
    'dribbles subis par 90': ('dribbles subis par 90 minutes', 'haut', 2), 'ballons perdus par 90': ('ballons perdus par 90 minutes', 'haut', 2),
    'fautes par 90': ('fautes par 90 minutes', 'haut', 2), 'erreurs menant à un tir par 90': ('erreurs menant à un tir par 90 minutes', 'haut', 2),
    '% duels aériens gagnés': ('% de duels aériens gagnés', 'bas', 1), '% duels gagnés': ('% de duels gagnés', 'bas', 1),
}
postes_pluriel = {'D': 'défenseurs', 'M': 'milieux', 'F': 'attaquants'}  # noms des postes pour les phrases


def points_individuels(equipe_us):  # joueurs de l'équipe parmi les 10 % les pires de leur poste, avec les règles validées
    v = agg[agg['id_equipe'] == us_vers_so[equipe_us]]  # joueurs de l'équipe avec assez de minutes
    lignes = []  # joueurs signalés
    for _, r in v.iterrows():  # parcourt les joueurs
        for ind, (texte_ind, cote, nb_dec) in libelles.items():  # parcourt les indicateurs
            if ind not in v.columns or pd.isna(r.get('mauvais ' + ind)) or r['mauvais ' + ind] < SEUIL_MAUVAIS:  # absent, vide ou pas assez mauvais
                continue  # on passe
            if ind.startswith('erreurs') and r['errorLeadToAShot'] < MIN_ERREURS:  # trop peu d'erreurs
                continue  # on ignore
            lignes.append({'joueur': r['joueur'], 'poste': r['poste'], 'indicateur': ind, 'texte': texte_ind, 'valeur': round(r[ind], nb_dec), 'minutes': int(r['minutes']), 'erreurs': int(r['errorLeadToAShot'])})  # une ligne par signalement
    return pd.DataFrame(lignes, columns=['joueur', 'poste', 'indicateur', 'texte', 'valeur', 'minutes', 'erreurs'])  # tableau des joueurs signalés

points_individuels(EQUIPE)  # lance pour l'équipe d'exemple : un joueur par ligne et par indicateur signalé

,joueur,poste,indicateur,texte,valeur,minutes,erreurs
0,Pervis Estupiñán,M,erreurs menant à un tir par 90,erreurs menant à un tir par 90 minutes,0.34,1046,4
1,Alexis Saelemaekers,M,erreurs menant à un tir par 90,erreurs menant à un tir par 90 minutes,0.13,2797,4
2,Youssouf Fofana,M,erreurs menant à un tir par 90,erreurs menant à un tir par 90 minutes,0.16,2217,4
3,Youssouf Fofana,M,% duels aériens gagnés,% de duels aériens gagnés,31.20,2217,4
4,Youssouf Fofana,M,% duels gagnés,% de duels gagnés,41.70,2217,4


## 20. Les pistes à tester

Un rapport d'adversaire doit aider à décider, sans prétendre décider à la place du staff. Chaque **piste** est donc attachée à un **chiffre précis** calculé plus haut. Il n'y a aucun plan de jeu général : si aucun chiffre ne soutient une idée, elle n'apparaît pas.

Quatre sources de pistes :
1. un **point faible d'équipe** de la partie 4 (par exemple beaucoup d'erreurs menant à un tir) donne une piste à tester (par exemple presser pour provoquer ces erreurs) ;
2. un **point fort d'équipe** donne un point à respecter ;
3. un **joueur parmi les 10 % les plus défavorables de son poste** donne un point d'appui possible, à confirmer sur son placement réel ;
4. un **joueur à surveiller**, ou une **dépendance à un joueur**, donne un point d'attention défensif.

Chaque ligne du résultat est présentée en deux temps : la **donnée** (un fait chiffré, comparé à la ligue) puis la **piste à tester** (une idée, jamais une certitude).

In [30]:
PISTES_FAIBLE = {  # point faible d'équipe : piste à tester
    'Subi : xG concédé': "Chercher des occasions de qualité : l'équipe concède beaucoup de xG.",
    'Subi : tirs concédés': "Multiplier les tirs : l'équipe en concède beaucoup.",
    'Subi : tirs concédés dans la surface': "Chercher à entrer dans la surface : c'est là qu'elle concède le plus.",
    'Subi : grosses occasions concédées': "Chercher les situations franches : elle en concède beaucoup.",
    'Défense : duels aériens gagnés (%)': "Tester le jeu aérien (centres, longs ballons) : elle gagne peu ses duels aériens.",
    'Défense : duels au sol gagnés (%)': "Chercher les duels au sol : elle les gagne peu.",
    'Défense : erreurs menant à un tir': "Presser pour provoquer ses erreurs : elle en commet beaucoup.",
    'Discipline : cartons jaunes': "Chercher les duels avec des joueurs déjà avertis : elle reçoit beaucoup de jaunes.",
    'Discipline : cartons rouges': "Jouer vite dans les espaces : elle se retrouve souvent à dix.",
}
PISTES_FORT = {  # point fort d'équipe : ce qu'il faut respecter
    'Subi : grosses occasions concédées': "Elle concède peu d'occasions franches : celles-ci seront à construire.",
    'Subi : xG concédé': "Peu de xG concédé : ses défenseurs limitent la qualité des tirs.",
    'Défense : duels aériens gagnés (%)': "Elle gagne ses duels aériens : les centres seront moins rentables.",
    'Défense : duels au sol gagnés (%)': "Elle gagne ses duels au sol : préférer des circuits de passes aux duels individuels.",
    'Attaque : centres réussis': "Elle réussit beaucoup de centres : défendre les côtés et la surface.",
    'Attaque : grosses occasions créées': "Elle crée beaucoup d'occasions franches : le plan défensif devra protéger la surface.",
    'Attaque : xG': "Elle crée des occasions de qualité : le plan défensif devra limiter ses tirs dans la surface.",
}

def pistes_de_jeu(equipe_us):  # fonction réutilisable : faits chiffrés et pistes à tester pour une équipe
    id_eq = us_vers_so[equipe_us]  # numéro Sofascore de l'équipe
    sortie = []  # liste de (fait, piste)
    concl = conclusions_ligue(id_eq)  # conclusions d'équipe validées (points forts, faibles, style)
    for r in concl.itertuples():  # parcourt les conclusions
        if r.type == 'Point faible' and r.indicateur in PISTES_FAIBLE:  # point faible avec piste connue
            sortie.append((r.texte, PISTES_FAIBLE[r.indicateur]))  # fait et piste
        if r.type == 'Point fort' and r.indicateur in PISTES_FORT:  # point fort à respecter
            sortie.append((r.texte, PISTES_FORT[r.indicateur]))  # fait et piste
    cpa_t = comparer_equipe(equipe_us, cpa_ligue[colonnes_cpa])  # penaltys et coups de pied arrêtés dans la ligue
    ligne_pen = cpa_t[cpa_t['indicateur'] == 'penalty subi : tirs'].iloc[0]  # penaltys concédés
    if ligne_pen['rang (1 = plus haut)'] <= 5:  # dans les 5 équipes qui en concèdent le plus
        sortie.append((f"Penaltys concédés : {int(ligne_pen['équipe'])} contre {str(ligne_pen['moyenne ligue']).replace('.', ',')} en moyenne dans la ligue (rang {int(ligne_pen['rang (1 = plus haut)'])} sur 20, petits nombres).", "Chercher des actions dans la surface plutôt que des frappes lointaines, en gardant en tête que l'écart repose sur peu d'événements."))  # fait et piste
    pi = points_individuels(equipe_us)  # joueurs signalés individuellement
    for nom, g in pi.groupby('joueur', sort=False):  # un bloc par joueur
        morceaux = []  # indicateurs du joueur
        for x in g.itertuples():  # parcourt ses signalements
            val = str(x.valeur).replace('.', ',')  # valeur avec la virgule française
            morceaux.append(f"{val} {x.texte}" + (f" ({x.erreurs} erreurs sur la saison)" if x.indicateur.startswith('erreurs') else ""))  # ajoute le nombre brut d'erreurs
        sortie.append((f"{nom} ({g.iloc[0]['poste']}) : {'; '.join(morceaux)}, parmi les 10 % les plus défavorables de son poste dans la ligue ({g.iloc[0]['minutes']} minutes).", f"Point d'appui possible : orienter le jeu ou le pressing vers {nom}, à confirmer sur son placement réel."))  # fait et piste
    for titre, lignes in a_surveiller(equipe_us):  # joueurs à surveiller
        for l in lignes:  # parcourt les phrases
            sortie.append((l, "Menace offensive établie : vérifier la composition annoncée." if titre == 'Menace principale' else "Peu de minutes mais bon taux de menace : à surveiller s'il entre en jeu."))  # fait et piste
    for p in conclusions_joueurs(equipe_us, joueurs_dangereux(equipe_us)):  # conclusions du bloc joueurs offensifs
        if p.startswith('Dépendance'):  # dépendance à un joueur
            sortie.append((p, "Concentrer l'attention défensive sur ce joueur, sans négliger le reste."))  # fait et piste
    return sortie  # liste des couples (fait, piste)

for i, (fait, piste) in enumerate(pistes_de_jeu(EQUIPE), 1):  # lance pour l'équipe d'exemple
    print(f"{i}. DONNÉE : {fait}\n   PISTE À TESTER : {piste}\n")  # affiche la donnée puis la piste

1. DONNÉE : Attaque : xG, 1,59 par match contre 1,26 par match en moyenne dans la ligue (4e valeur la plus haute sur 20).
   PISTE À TESTER : Elle crée des occasions de qualité : le plan défensif devra limiter ses tirs dans la surface.

2. DONNÉE : Attaque : grosses occasions créées, 2,63 par match contre 2,07 par match en moyenne dans la ligue (3e valeur la plus haute sur 20).
   PISTE À TESTER : Elle crée beaucoup d'occasions franches : le plan défensif devra protéger la surface.

3. DONNÉE : Attaque : centres réussis, 5,08 par match contre 4,26 par match en moyenne dans la ligue (2e valeur la plus haute sur 20).
   PISTE À TESTER : Elle réussit beaucoup de centres : défendre les côtés et la surface.

4. DONNÉE : Défense : duels au sol gagnés (%), 52,21 % contre 50,0 % en moyenne dans la ligue (2e valeur la plus haute sur 20).
   PISTE À TESTER : Elle gagne ses duels au sol : préférer des circuits de passes aux duels individuels.

5. DONNÉE : Défense : erreurs menant à un tir, 0,76 p

## 21. La dynamique de la saison

Les parties précédentes décrivent la saison entière. Cette partie regarde comment elle évolue, avec cinq blocs, tous calculés match par match à partir des données Understat :
1. **Domicile et extérieur** : les points par match de l'équipe sur chaque terrain, comparés à l'avantage du terrain habituel de la ligue (seuil de 0,30 point par match d'écart).
2. **Forme récente** : les 5 derniers matchs comparés à la saison (seuil de 0,5 point par match pour parler de progression ou de baisse).
3. **Style de jeu** : l'étiquette et le PPDA vus dans la partie 1.
4. **Résultats selon le style de l'adversaire** : l'équipe prend elle moins de points contre les blocs bas, les pressings médians ou les pressings hauts ? Un groupe d'adversaires doit compter au moins 10 matchs pour être pris en compte, avec un écart d'au moins 0,4 point par match avec le reste.
5. **Moment des buts** : buts marqués et encaissés par tranche de 15 minutes. La dernière tranche inclut les arrêts de jeu.

Pour relier chaque match à son adversaire, la fonction croise la date, le lieu et le xG du match. Elle s'arrête avec une erreur claire si une liaison est incomplète, plutôt que de produire un résultat faux.

**Contrôles attendus pour Milan** : 38 matchs, 70 points, 53 buts marqués et 35 encaissés dans les tirs, et les buts par tranche (5,5) (4,7) (12,5) (13,7) (6,5) (13,6).

In [31]:
colonnes_utiles = ['date', 'h_a', 'scored', 'missed', 'result', 'pts', 'xG', 'xGA']  # colonnes gardées pour les blocs de résultats et de forme
SEUIL_FORME = 0.5  # écart de points (ou de buts) par match à partir duquel on signale une évolution de la forme
SEUIL_GROUPE = 0.4  # écart de points par match à partir duquel on signale une difficulté contre un style d'adversaire
MIN_MATCHS_GROUPE = 10  # nombre minimum de matchs pour qu'un groupe d'adversaires soit pris en compte
SEUIL_ECART_LIEU = 0.30  # écart, en points par match, par rapport à la moyenne de la ligue, pour signaler une différence domicile et extérieur
N_FORME = 5  # nombre de derniers matchs regardés pour la forme récente
noms_style = {'pressing haut': 'Pressing haut', 'pressing median': 'Pressing médian', 'bloc bas': 'Bloc bas'}  # noms d'affichage des étiquettes de style
bornes = [0, 15, 30, 45, 60, 75, 130]  # limites des tranches de 15 minutes, la dernière couvre les arrêts de jeu
noms_tranches = ['0 à 15', '16 à 30', '31 à 45', '46 à 60', '61 à 75', '76 et plus']  # noms affichés pour chaque tranche

def tableau_matchs(equipe):  # reproduit les sections 3 et 5 pour n'importe quelle équipe : ses matchs avec l'adversaire de chacun
    info = [v for v in league_table.values() if v['title'] == equipe][0]  # données Understat de l'équipe
    m = pd.DataFrame(info['history'])[colonnes_utiles].copy()  # historique de matchs, colonnes utiles seulement
    m = m.rename(columns={'h_a': 'lieu', 'scored': 'buts_marques', 'missed': 'buts_encaisses', 'result': 'resultat', 'pts': 'points'})  # noms en français
    m['date'] = pd.to_datetime(m['date'])  # date en vraie date
    for col in ['buts_marques', 'buts_encaisses', 'points', 'xG', 'xGA']:  # colonnes numériques
        m[col] = pd.to_numeric(m[col])  # convertit en nombres
    m['lieu'] = m['lieu'].map({'h': 'domicile', 'a': 'exterieur'})  # mots clairs
    m = m.sort_values('date').reset_index(drop=True)  # du plus ancien au plus récent
    autres = tous_matchs[tous_matchs['equipe'] != equipe].copy()  # matchs des 19 autres équipes, donc ceux où notre équipe est l'adversaire
    autres['lieu_equipe'] = autres['h_a'].map({'h': 'exterieur', 'a': 'domicile'})  # si l'autre jouait à domicile, notre équipe jouait dehors
    autres['xG_cle'] = autres['xGA'].round(2)  # xG concédés par l'autre équipe = xG de la nôtre, clé de liaison
    m['xG_cle'] = m['xG'].round(2)  # même clé côté équipe
    m = m.merge(autres[['date', 'lieu_equipe', 'xG_cle', 'equipe']].rename(columns={'equipe': 'adversaire'}), left_on=['date', 'lieu', 'xG_cle'], right_on=['date', 'lieu_equipe', 'xG_cle'], how='left')  # relie chaque match à l'adversaire
    m = m.drop(columns=['xG_cle', 'lieu_equipe'])  # retire les colonnes de liaison
    if len(m) != len(info['history']) or m['adversaire'].isna().any():  # contrôle de la liaison : aucun doublon, aucun adversaire manquant
        raise ValueError(f"{equipe} : liaison des adversaires incomplète ({len(m)} matchs, {int(m['adversaire'].isna().sum())} sans adversaire)")  # on s'arrête avec un message clair
    m['victoire'] = (m['resultat'] == 'w').astype(int)  # 1 si victoire
    m['nul'] = (m['resultat'] == 'd').astype(int)  # 1 si nul
    m['defaite'] = (m['resultat'] == 'l').astype(int)  # 1 si défaite
    return m  # tableau de matchs de l'équipe

def historique_equipe(equipe):  # blocs « dynamique » pour une équipe : lieu, forme, style, résultats selon l'adversaire, moment des buts
    m = tableau_matchs(equipe)  # matchs de l'équipe
    sortie = {'phrases': {}}  # résultat et phrases de conclusion

    # Domicile et extérieur
    bl = m.groupby('lieu').agg(matchs=('date', 'count'), victoires=('victoire', 'sum'), nuls=('nul', 'sum'), defaites=('defaite', 'sum'), points=('points', 'sum'), buts_marques=('buts_marques', 'sum'), buts_encaisses=('buts_encaisses', 'sum'), xG=('xG', 'sum'), xGA=('xGA', 'sum')).round(1)  # bilan par lieu
    bl['points_par_match'] = (bl['points'] / bl['matchs']).round(2)  # points par match
    sortie['lieu'] = json.loads(bl.reset_index().to_json(orient='records', force_ascii=False))  # tableau pour le site
    moy_lieu = ecart_lieu['ecart_ext_moins_dom'].mean()  # avantage du terrain habituel de la ligue
    dif = ecart_lieu.loc[equipe, 'ecart_ext_moins_dom'] - moy_lieu  # écart de l'équipe à la normale
    p_dom, p_ext = ecart_lieu.loc[equipe, 'domicile'], ecart_lieu.loc[equipe, 'exterieur']  # points par match
    nb_dehors = int((ecart_lieu['ecart_ext_moins_dom'] > 0).sum())  # équipes qui font mieux dehors
    if dif >= SEUIL_ECART_LIEU:  # nettement plus à l'aise dehors
                sortie['phrases']['domicile_exterieur'] = f"Plus à l'aise à l'extérieur : {fr(p_ext)} points par match contre {fr(p_dom)} à domicile, un cas rare puisque seulement {nb_dehors} équipes sur 20 prennent plus de points dehors qu'à domicile."  # phrase
    elif dif <= -SEUIL_ECART_LIEU:  # nettement plus à l'aise à domicile
        sortie['phrases']['domicile_exterieur'] = f"Plus à l'aise à domicile : {fr(p_dom)} points par match contre {fr(p_ext)} à l'extérieur, un avantage du terrain plus marqué que la moyenne de la Serie A ({fr(-moy_lieu)} point par match)."  # phrase
    else:  # dans la normale
        sortie['phrases']['domicile_exterieur'] = f"Équilibré par rapport à la Serie A : {fr(p_dom)} points par match à domicile et {fr(p_ext)} à l'extérieur."  # phrase

    # Forme récente
    der = m.tail(N_FORME)  # N derniers matchs
    cols = {'points_par_match': 'points', 'buts_marques_par_match': 'buts_marques', 'buts_encaisses_par_match': 'buts_encaisses', 'xG_par_match': 'xG', 'xGA_par_match': 'xGA'}  # nom du résultat : colonne source
    rec = {k: round(der[v].mean(), 2) for k, v in cols.items()}  # moyennes sur la forme récente
    sai = {k: round(m[v].mean(), 2) for k, v in cols.items()}  # moyennes sur la saison
    sortie['forme'] = {'sequence': [x.upper() for x in der['resultat']], 'recent': rec, 'saison': sai, 'n': N_FORME}  # données pour le site
    ecart_pts = rec['points_par_match'] - sai['points_par_match']  # écart de points par match
    tendance = 'en progression' if ecart_pts >= SEUIL_FORME else ('en baisse' if ecart_pts <= -SEUIL_FORME else 'stable')  # tendance de la forme
    details = []  # remarques complémentaires
    if rec['buts_encaisses_par_match'] - sai['buts_encaisses_par_match'] >= SEUIL_FORME:  # défense plus fragile
        details.append(f"défense plus fragile ({fr(rec['buts_encaisses_par_match'])} but encaissé par match contre {fr(sai['buts_encaisses_par_match'])})")  # remarque
    if sai['buts_marques_par_match'] - rec['buts_marques_par_match'] >= SEUIL_FORME:  # attaque moins efficace
        details.append(f"attaque moins efficace ({fr(rec['buts_marques_par_match'])} but marqué par match contre {fr(sai['buts_marques_par_match'])})")  # remarque
    phrase = f"Forme {tendance} : {fr(rec['points_par_match'])} points par match sur les {N_FORME} derniers matchs contre {fr(sai['points_par_match'])} sur la saison"  # phrase
    sortie['phrases']['forme'] = phrase + (", avec " + " et ".join(details) if details else "") + "."  # termine la phrase

    # Style de jeu
    lg = style[style['equipe'] == equipe].iloc[0]  # ligne de style de l'équipe
    sortie['style'] = {'etiquette': lg['etiquette'], 'ppda': float(lg['ppda']), 'deep_subi': float(lg['deep_subi_par_match']), 'rang_ppda': int(style.index[style['equipe'] == equipe][0])}  # données pour le site
    sortie['phrases']['style'] = f"{noms_style[lg['etiquette']]} : PPDA de {fr(lg['ppda'])} et {fr(lg['deep_subi_par_match'])} actions subies près de la surface par match."  # phrase

    # Résultats selon le style de l'adversaire
    m['style_adversaire'] = m['adversaire'].map(style.set_index('equipe')['etiquette'])  # style de l'adversaire de chaque match
    bs = m.groupby('style_adversaire').agg(matchs=('date', 'count'), points=('points', 'sum'), buts_marques=('buts_marques', 'sum'), buts_encaisses=('buts_encaisses', 'sum'), xG=('xG', 'sum'), xGA=('xGA', 'sum')).round(1)  # bilan par style
    bs['points_par_match'] = (bs['points'] / bs['matchs']).round(2)  # points par match
    bs = bs.reindex(['pressing haut', 'pressing median', 'bloc bas']).dropna(subset=['matchs'])  # ordre logique, sans groupe vide
    sortie['selon_adversaire'] = json.loads(bs.reset_index().to_json(orient='records', force_ascii=False))  # tableau pour le site
    valides = bs[bs['matchs'] >= MIN_MATCHS_GROUPE]  # groupes assez gros
    phrase_sa = "Résultats réguliers quel que soit le style de l'adversaire."  # phrase par défaut
    if len(valides):  # s'il existe au moins un groupe assez gros
        pire = valides['points_par_match'].idxmin()  # style contre lequel l'équipe prend le moins de points
        autres_pts = m[m['style_adversaire'] != pire]['points'].mean()  # points par match contre les autres styles
        if autres_pts - bs.loc[pire, 'points_par_match'] >= SEUIL_GROUPE:  # écart assez grand
            phrase_sa = f"Moins à l'aise contre les adversaires en {noms_style[pire].lower()} : {fr(bs.loc[pire, 'points_par_match'])} points par match ({int(bs.loc[pire, 'matchs'])} matchs) contre {fr(autres_pts)} face aux autres."  # phrase
    sortie['phrases']['selon_adversaire'] = phrase_sa  # range la phrase

    # Moment des buts, à partir des tirs de la ligue
    t = tirs_ligue[(tirs_ligue['equipe'] == equipe) | (tirs_ligue['adversaire'] == equipe)].copy()  # tirs des matchs de l'équipe
    t['but'] = t['result'].isin(['Goal', 'OwnGoal']).astype(int)  # 1 si but, buts contre son camp compris
    t['pour'] = (t['equipe_but'] == equipe)  # but bénéficiant à l'équipe
    t['tranche'] = pd.cut(t['minute'], bins=bornes, labels=noms_tranches, include_lowest=True)  # tranche de 15 minutes
    bt = pd.DataFrame({  # une ligne par tranche
        'buts_marques': t[t['pour']].groupby('tranche', observed=False)['but'].sum(),  # buts marqués
        'buts_encaisses': t[~t['pour']].groupby('tranche', observed=False)['but'].sum(),  # buts encaissés
        'xG': t[t['equipe'] == equipe].groupby('tranche', observed=False)['xG'].sum().round(1),  # qualité des occasions créées
        'xGA': t[t['adversaire'] == equipe].groupby('tranche', observed=False)['xG'].sum().round(1),  # qualité des occasions concédées
    }).reindex(noms_tranches).fillna(0)  # ordre des tranches
    bt['difference_buts'] = bt['buts_marques'] - bt['buts_encaisses']  # écart de buts
    sortie['moment_buts'] = json.loads(bt.reset_index().rename(columns={'index': 'tranche'}).to_json(orient='records', force_ascii=False))  # tableau pour le site
    frag = str(bt['difference_buts'].idxmin())  # tranche la plus défavorable
    meilleur_ecart = bt['difference_buts'].max()  # meilleur écart de buts
    fortes = ", ".join(f"minutes {x} ({int(bt.loc[x, 'buts_marques'])} contre {int(bt.loc[x, 'buts_encaisses'])})" for x in bt.index[bt['difference_buts'] == meilleur_ecart])  # tranches fortes
    if bt.loc[frag, 'difference_buts'] < 0:  # une tranche défavorable existe
        sortie['phrases']['moment_des_buts'] = f"Période la plus fragile (minutes {frag}) : {int(bt.loc[frag, 'buts_marques'])} buts marqués pour {int(bt.loc[frag, 'buts_encaisses'])} encaissés. Périodes les plus fortes : {fortes}."  # phrase
    else:  # aucune tranche défavorable
        sortie['phrases']['moment_des_buts'] = f"Aucune période où l'équipe encaisse plus qu'elle ne marque. Périodes les plus fortes : {fortes}."  # phrase
    sortie['controle'] = {'matchs': int(len(m)), 'points': int(m['points'].sum()), 'buts_marques_tirs': int(bt['buts_marques'].sum()), 'buts_encaisses_tirs': int(bt['buts_encaisses'].sum())}  # chiffres de contrôle
    return sortie  # dictionnaire complet

hist_milan = historique_equipe(EQUIPE)  # test sur l'équipe d'exemple, dont on connaît les résultats attendus
print("Contrôle Milan :", hist_milan['controle'], "| attendu : 38 matchs, 70 points, 53 buts marqués et 35 encaissés")  # contrôle principal
print("Moment des buts (marqués, encaissés) :", [(int(a['buts_marques']), int(a['buts_encaisses'])) for a in hist_milan['moment_buts']], "| attendu : (5,5) (4,7) (12,5) (13,7) (6,5) (13,6)")  # contrôle par tranche
for cle, texte in hist_milan['phrases'].items():  # affiche les phrases
    print("•", texte)  # une phrase par ligne

Contrôle Milan : {'matchs': 38, 'points': 70, 'buts_marques_tirs': 53, 'buts_encaisses_tirs': 35} | attendu : 38 matchs, 70 points, 53 buts marqués et 35 encaissés
Moment des buts (marqués, encaissés) : [(5, 5), (4, 7), (12, 5), (13, 7), (6, 5), (13, 6)] | attendu : (5,5) (4,7) (12,5) (13,7) (6,5) (13,6)
• Plus à l'aise à l'extérieur : 2,00 points par match contre 1,68 à domicile, un cas rare puisque seulement 6 équipes sur 20 prennent plus de points dehors qu'à domicile.
• Forme en baisse : 0,80 points par match sur les 5 derniers matchs contre 1,84 sur la saison, avec défense plus fragile (1,60 but encaissé par match contre 0,92).
• Bloc bas : PPDA de 14,62 et 5,32 actions subies près de la surface par match.
• Moins à l'aise contre les adversaires en bloc bas : 1,50 points par match (14 matchs) contre 2,04 face aux autres.
• Période la plus fragile (minutes 16 à 30) : 4 buts marqués pour 7 encaissés. Périodes les plus fortes : minutes 31 à 45 (12 contre 5), minutes 76 et plus (13 co

## 22. Le rapport complet d'une équipe, puis des 20 équipes

La fonction `analyser_equipe` rassemble tout ce qui précède pour **une** équipe : comparaison avec la ligue, points forts et points faibles, coups de pied arrêtés, joueurs dangereux, effectif, joueurs à surveiller, pistes et dynamique de la saison. Le résultat est un dictionnaire prêt à être lu par le site.

On la lance ensuite sur les 20 équipes. Si une équipe provoque une erreur, elle est notée et les autres continuent : on ne perd pas tout pour un seul problème.

**Contrôle attendu** : 20 rapports produits et 0 échec.

In [32]:
import json, math  # pour sauvegarder en JSON et gérer les valeurs vides

def propre(df):  # transforme un tableau en liste de dictionnaires sans valeurs vides (le JSON n'accepte pas NaN)
    return json.loads(df.to_json(orient='records', force_ascii=False))  # to_json remplace les NaN par null

def analyser_equipe(equipe_us):  # rapport complet d'une équipe, sous forme de dictionnaire prêt pour le site
    id_eq = us_vers_so[equipe_us]  # numéro Sofascore de l'équipe
    rang_classement = equipes_sofa.set_index('id_equipe').loc[id_eq]  # place et points de la saison
    fiche_eq = fiche_joueurs(equipe_us)  # minutes, rôle et menace des joueurs
    return {  # tout ce que le site affichera
        'equipe': equipe_us,  # nom Understat
        'nom_sofascore': rang_classement['equipe'],  # nom Sofascore
        'classement': int(rang_classement['rang']), 'points': int(rang_classement['points']),  # place et points
        'comparaison_ligue': propre(comparer_equipe(id_eq)),  # indicateurs d'équipe contre la ligue
        'conclusions_ligue': propre(conclusions_ligue(id_eq)),  # points forts, faibles et style
        'coups_de_pied_arretes': propre(comparer_equipe(equipe_us, cpa_ligue[colonnes_cpa])),  # penaltys et coups de pied arrêtés contre la ligue
        'joueurs_dangereux': propre(joueurs_dangereux(equipe_us).reset_index().head(10)),  # les 10 joueurs les plus impliqués
        'phrases_joueurs': conclusions_joueurs(equipe_us, joueurs_dangereux(equipe_us)),  # phrases du bloc joueurs offensifs
        'effectif': propre(fiche_eq.reset_index()),  # minutes, rôle, menace
        'a_surveiller': [{'titre': t, 'phrases': l} for t, l in a_surveiller(equipe_us)],  # liste à surveiller
        'pistes': [{'donnee': f, 'piste': p} for f, p in pistes_de_jeu(equipe_us)],  # pistes à tester
        'historique': historique_equipe(equipe_us),  # dynamique de la saison : lieu, forme, style, résultats selon l'adversaire, moment des buts
    }

rapports, echecs = {}, {}  # résultats et erreurs par équipe
for equipe in correspondance_equipes['understat']:  # parcourt les 20 équipes
    try:  # on attrape l'erreur pour ne pas bloquer les autres équipes
        rapports[equipe] = analyser_equipe(equipe)  # rapport de l'équipe
    except Exception as e:  # si un bloc plante
        echecs[equipe] = f"{type(e).__name__} : {e}"  # on note l'erreur

print("Rapports produits :", len(rapports), "sur 20 | échecs :", len(echecs))  # contrôle : on attend 20 et 0
for eq, msg in echecs.items():  # affiche les échecs
    print(" -", eq, ":", msg)  # nom et erreur
resume = pd.DataFrame([{'équipe': e, 'classement': r['classement'], 'points forts': sum(c['type'] == 'Point fort' for c in r['conclusions_ligue']), 'points faibles': sum(c['type'] == 'Point faible' for c in r['conclusions_ligue']), 'pistes': len(r['pistes']), 'joueurs à surveiller': sum(len(b['phrases']) for b in r['a_surveiller'])} for e, r in rapports.items()]).sort_values('classement')  # résumé par équipe pour contrôle visuel
os.makedirs('data/processed', exist_ok=True)  # dossier de sortie
with open('data/processed/rapports_equipes.json', 'w', encoding='utf-8') as f:  # fichier qui alimentera le site
    json.dump(rapports, f, ensure_ascii=False, indent=1)  # sauvegarde des 20 rapports
resume  # affiche le résumé

Rapports produits : 20 sur 20 | échecs : 0


,équipe,classement,points forts,points faibles,pistes,joueurs à surveiller
0,Inter,1,16,0,13,4
1,Napoli,2,8,1,18,5
2,Roma,3,8,2,15,6
3,Como,4,11,1,20,4
4,AC Milan,5,6,1,15,5
5,Juventus,6,16,1,14,5
6,Atalanta,7,9,0,17,5
7,Bologna,8,3,2,15,5
8,Lazio,9,1,5,13,4
9,Udinese,10,2,2,13,5


## 23. Contrôle de cohérence

Avant de publier, on vérifie que le rapport ne raconte pas n'importe quoi. Test simple : le solde « points forts moins points faibles » de chaque équipe doit aller dans le même sens que ses points au classement, sans être identique, puisque ce solde n'est pas un score. On mesure cet accord avec une corrélation de rang (Spearman), qui vaut 1 si les deux classements sont identiques et 0 s'ils n'ont aucun lien. On regarde ensuite deux équipes à la main, Sassuolo et la Juventus, pour lire les phrases et juger si elles sont plausibles.

**Contrôle attendu** : une corrélation nettement positive (0,81 dans notre version).

In [33]:
tab = resume.merge(equipes_sofa[['rang', 'points']].rename(columns={'rang': 'classement'}), on='classement')  # ajoute les points de chaque équipe
tab['forts moins faibles'] = tab['points forts'] - tab['points faibles']  # solde des points forts et faibles
print("Corrélation (rangs) entre solde forts moins faibles et points :", round(tab['forts moins faibles'].corr(tab['points'], method='spearman'), 2))  # on attend nettement positive
for eq in ['Sassuolo', 'Juventus']:  # les deux cas à examiner
    print(f"\n--- {eq} ---")  # titre
    for c in rapports[eq]['conclusions_ligue']:  # parcourt ses conclusions d'équipe
        print(f"[{c['type']}] {c['texte']}")  # type et phrase

Corrélation (rangs) entre solde forts moins faibles et points : 0.81

--- Sassuolo ---
[Style] Style : entrées dans le dernier tiers, 47,89 par match contre 51,47 par match en moyenne dans la ligue (5e valeur la plus basse sur 20).
[Style] Style : longs ballons réussis, 20,87 par match contre 22,56 par match en moyenne dans la ligue (5e valeur la plus basse sur 20).
[Point faible] Attaque : tirs, 10,66 par match contre 12,31 par match en moyenne dans la ligue (4e valeur la plus basse sur 20).
[Point faible] Attaque : centres réussis, 3,39 par match contre 4,26 par match en moyenne dans la ligue (3e valeur la plus basse sur 20).
[Style] Défense : tacles, 13,63 par match contre 15,19 par match en moyenne dans la ligue (4e valeur la plus basse sur 20).
[Point faible] Défense : duels aériens gagnés (%), 47,76 % contre 50,0 % en moyenne dans la ligue (5e valeur la plus basse sur 20).
[Point faible] Défense : duels au sol gagnés (%), 48,89 % contre 50,0 % en moyenne dans la ligue (5e valeur 

## 24. Le site

Le site est une seule page HTML, sans serveur. On injecte les 20 rapports dans un modèle de page (`docs/modele.html`), et le fichier produit (`docs/index.html`) s'ouvre dans n'importe quel navigateur. On choisit une équipe dans le menu, et la page affiche son rapport avec graphiques, points forts et points faibles. L'adresse accepte un paramètre (`?equipe=`) pour partager le lien direct vers une équipe.

Le modèle `docs/modele.html` doit être présent dans le dossier `docs` avant de lancer la cellule.

**Contrôle attendu** : 20 équipes dans le fichier, puis la page créée.

In [35]:
import json, os  # pour lire le modèle de page et sauvegarder les fichiers du site

os.makedirs('docs', exist_ok=True)  # crée le dossier du site s'il n'existe pas encore
with open('data/processed/rapports_equipes.json', encoding='utf-8') as f:  # relit les 20 rapports sauvegardés
    donnees = json.load(f)  # dictionnaire : une entrée par équipe
print("Équipes dans le fichier :", len(donnees), "| attendu 20")  # contrôle : on attend 20 équipes

with open('docs/modele.html', encoding='utf-8') as f:  # ouvre le modèle de page (à placer dans le dossier docs)
    modele = f.read()  # lit tout le modèle
assert '/*DONNEES*/null' in modele, "Le marqueur de données est introuvable dans le modèle"  # arrête avec un message clair si le mauvais fichier est utilisé

json_texte = json.dumps(donnees, ensure_ascii=False).replace('</', '<\\/')  # données en texte, avec protection de la balise de fin de script
page = modele.replace('/*DONNEES*/null', json_texte)  # insère les données dans la page
with open('docs/index.html', 'w', encoding='utf-8') as f:  # écrit la page finale
    f.write(page)  # sauvegarde
print("Page créée : docs/index.html |", round(os.path.getsize('docs/index.html') / 1024), "ko")  # contrôle de la taille du fichier

Équipes dans le fichier : 20 | attendu 20
Page créée : docs/index.html | 641 ko


## Limites et précautions de lecture

1. **Les chiffres décrivent, ils ne prouvent pas.** Le rapport propose des pistes à tester. Il ne dit pas ce qui se passera le jour du match.
2. **Pas de score.** Compter les points forts et les points faibles n'a pas de sens entre équipes : plusieurs indicateurs racontent la même chose.
3. **Petits nombres.** Les penaltys, les erreurs individuelles et les joueurs de moins de 900 minutes reposent sur peu d'événements. Les phrases le disent.
4. **Le xG d'Understat surestime les buts d'environ 17 % sur toute la ligue.** Il sert à comparer les équipes entre elles, pas à dire qu'une équipe est « en sur performance ». Pour le xG d'équipe, on utilise celui de Sofascore (écart d'environ 4 %).
5. **Les seuils sont des choix de lecture** (PPDA, forme, domicile et extérieur, dépendance, minutes). Ils sont écrits en haut de chaque partie et modifiables.
6. **Une saison, un seul championnat.** Les comparaisons sont valables dans la Serie A 2025/26 uniquement.
7. **Aucune vidéo.** Un chiffre ne dit rien du placement, des consignes ou des absents du jour : le staff complète le rapport avec ce qu'il voit.